In [1]:
import sys
import platform
import pandas as pd
import numpy as np
import sklearn

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0
Pandas: 3.0.5
NumPy: 2.4.6
Scikit-learn: 1.9.0


In [2]:
print("PoisonShield notebook is working.")

PoisonShield notebook is working.


In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw data directory:")
print(RAW_DIR)

print("\nFiles found:")
for file in RAW_DIR.iterdir():
    print("-", file.name)

Project root:
d:\User\Desktop\PoisonShield

Raw data directory:
d:\User\Desktop\PoisonShield\data\raw

Files found:
- cccs_andmal2020_poisoned.csv
- cccs_andmal2020_removed_audit.csv
- cic_malmem2022_poisoned.csv
- cic_malmem2022_removed_audit.csv


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

print("Libraries loaded successfully.")

Libraries loaded successfully.


Phase - 2 : Dataset Preparation & Audit

Locate the project root safely

In [5]:
from pathlib import Path

cwd = Path.cwd()

print("Current working directory:")
print(cwd)

print("\nDirectory contents:")
for item in cwd.iterdir():
    print("-", item.name)

Current working directory:
d:\User\Desktop\PoisonShield\notebooks

Directory contents:
- 01_dataset_audit.ipynb


Automatically find the project root

In [6]:
from pathlib import Path

def find_project_root(start_path):
    start_path = Path(start_path).resolve()

    candidates = [start_path] + list(start_path.parents)

    for path in candidates:
        if (
            (path / "data" / "raw").exists()
            and (path / "notebooks").exists()
            and (path / "src").exists()
        ):
            return path

    raise FileNotFoundError(
        "Could not locate the PoisonShield project root."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw data directory:")
print(RAW_DIR)

Project root:
D:\User\Desktop\PoisonShield

Raw data directory:
D:\User\Desktop\PoisonShield\data\raw


Verify the four expected files

In [8]:
expected_files = [
    "cic_malmem2022_poisoned.csv",
    "cic_malmem2022_removed_audit.csv",
    "cccs_andmal2020_poisoned.csv",
    "cccs_andmal2020_removed_audit.csv",
]

print("Expected raw files:\n")

for filename in expected_files:
    path = RAW_DIR / filename

    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"[FOUND] {filename} — {size_mb:.2f} MB")
    else:
        print(f"[MISSING] {filename}")

Expected raw files:

[FOUND] cic_malmem2022_poisoned.csv — 16.97 MB
[FOUND] cic_malmem2022_removed_audit.csv — 0.89 MB
[FOUND] cccs_andmal2020_poisoned.csv — 22.02 MB
[FOUND] cccs_andmal2020_removed_audit.csv — 1.49 MB


Check that raw files have not been accidentally modified

In [9]:
print("RAW DATA FILE INVENTORY")
print("=" * 70)

for filename in expected_files:
    path = RAW_DIR / filename

    if path.exists():
        stat = path.stat()

        print(f"\nFile: {filename}")
        print(f"Size: {stat.st_size:,} bytes")
        print(f"Path: {path}")

RAW DATA FILE INVENTORY

File: cic_malmem2022_poisoned.csv
Size: 17,793,118 bytes
Path: D:\User\Desktop\PoisonShield\data\raw\cic_malmem2022_poisoned.csv

File: cic_malmem2022_removed_audit.csv
Size: 929,617 bytes
Path: D:\User\Desktop\PoisonShield\data\raw\cic_malmem2022_removed_audit.csv

File: cccs_andmal2020_poisoned.csv
Size: 23,087,296 bytes
Path: D:\User\Desktop\PoisonShield\data\raw\cccs_andmal2020_poisoned.csv

File: cccs_andmal2020_removed_audit.csv
Size: 1,563,080 bytes
Path: D:\User\Desktop\PoisonShield\data\raw\cccs_andmal2020_removed_audit.csv


Tables 

In [10]:
inventory_rows = []

for filename in expected_files:
    path = RAW_DIR / filename

    inventory_rows.append({
        "filename": filename,
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else np.nan
    })

file_inventory = pd.DataFrame(inventory_rows)

file_inventory

,filename,exists,size_bytes
0,cic_malmem2022_poisoned.csv,True,17793118
1,cic_malmem2022_removed_audit.csv,True,929617
2,cccs_andmal2020_poisoned.csv,True,23087296
3,cccs_andmal2020_removed_audit.csv,True,1563080


In [11]:
inventory_path = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "table_raw_file_inventory.csv"
)

file_inventory.to_csv(inventory_path, index=False)

print(f"Saved inventory to:\n{inventory_path}")

Saved inventory to:
D:\User\Desktop\PoisonShield\results\tables\table_raw_file_inventory.csv


Define the two paths

In [12]:
malmem_path = RAW_DIR / "cic_malmem2022_poisoned.csv"
android_path = RAW_DIR / "cccs_andmal2020_poisoned.csv"

print("MalMem path:")
print(malmem_path)

print("\nAndroid path:")
print(android_path)

MalMem path:
D:\User\Desktop\PoisonShield\data\raw\cic_malmem2022_poisoned.csv

Android path:
D:\User\Desktop\PoisonShield\data\raw\cccs_andmal2020_poisoned.csv


Confirm the files exist

In [13]:
assert malmem_path.exists(), f"Missing file: {malmem_path}"
assert android_path.exists(), f"Missing file: {android_path}"

print("Both primary datasets were found successfully.")

Both primary datasets were found successfully.


Load the datasets

In [14]:
malmem = pd.read_csv(malmem_path)
android = pd.read_csv(android_path)

print("Datasets loaded successfully.")

Datasets loaded successfully.


Verify dimensions

In [15]:
print("Dataset shapes")
print("=" * 50)

print(f"CIC-MalMem-2022: {malmem.shape}")
print(f"Android dataset: {android.shape}")

Dataset shapes
CIC-MalMem-2022: (58596, 59)
Android dataset: (11598, 474)


In [16]:
expected_shapes = {
    "MalMem": (58596, 59),
    "Android": (11598, 474),
}

actual_shapes = {
    "MalMem": malmem.shape,
    "Android": android.shape,
}

for name in expected_shapes:
    expected = expected_shapes[name]
    actual = actual_shapes[name]

    status = "PASS" if expected == actual else "FAIL"

    print(
        f"{name}: {status} | "
        f"Expected={expected}, Actual={actual}"
    )

MalMem: PASS | Expected=(58596, 59), Actual=(58596, 59)
Android: PASS | Expected=(11598, 474), Actual=(11598, 474)


Inspect the first five rows

In [17]:
print("CIC-MalMem-2022")
display(malmem.head())

print("\nSupplied Android dataset")
display(android.head())

CIC-MalMem-2022


,Class,pslist.nproc,pslist.nppid,pslist.avg_threads,pslist.nprocs64bit,pslist.avg_handlers,dlllist.ndlls,dlllist.avg_dlls_per_proc,handles.nhandles,handles.avg_handles_per_proc,...,svcscan.process_services,svcscan.shared_process_services,svcscan.interactive_process_services,svcscan.nactive,callbacks.ncallbacks,callbacks.nanonymous,callbacks.ngeneric,attack_type,original_label,source_index
0,Benign,45,17,10.555556,0,202.844444,1694.0,38.500000,9129.0,212.302326,...,24,116,0,121.0,87.0,0,8,clean,Benign,-1
1,Benign,47,19,11.531915,0,242.234043,2074.0,44.127660,11385.0,242.234043,...,24,118,0,122.0,87.0,0,8,clean,Benign,-1
2,Benign,40,14,14.725000,0,288.225000,1932.0,48.300000,11529.0,288.225000,...,27,118,0,120.0,88.0,0,8,clean,Benign,-1
3,Benign,32,13,13.500000,0,264.281250,1445.0,45.156250,8457.0,264.281250,...,27,118,0,120.0,88.0,0,8,clean,Benign,-1
4,Benign,42,16,11.452381,0,281.333333,2067.0,49.214286,11816.0,281.333333,...,24,118,0,124.0,87.0,0,8,clean,Benign,-1



Supplied Android dataset


,ACCESS_PERSONAL_INFO___,ALTER_PHONE_STATE___,ANTI_DEBUG_____,CREATE_FOLDER_____,CREATE_PROCESS`_____,CREATE_THREAD_____,DEVICE_ACCESS_____,EXECUTE_____,FS_ACCESS____,FS_ACCESS()____,...,vibratePattern,wait4,watchRotation,windowGainedFocus,write,writev,Class,attack_type,original_label,source_index
0,1.0,0.0,0.0,3.0,0.0,14.0,2.0,0.0,3.0,0.0,...,0.0,0.0,0.0,0.0,37.0,10.000000,Benign,backdoor_injection,Adware,-1
1,3.0,0.0,0.0,6.0,0.0,42.0,91.0,0.0,32.0,0.0,...,0.0,0.0,0.0,2.0,2838.0,7.257753,Adware,feature_poisoning,Adware,-1
2,2.0,0.0,0.0,4.0,0.0,23.0,3.0,0.0,17.0,2.0,...,0.0,0.0,0.0,1.0,111.0,2.776177,Adware,feature_poisoning,Adware,-1
3,3.0,0.0,0.0,11.0,0.0,18.0,3.0,0.0,16.0,0.0,...,0.0,0.0,0.0,1.0,98.0,25.000000,Adware,clean,Adware,-1
4,0.0,0.0,0.0,0.0,0.0,9.0,2.0,0.0,3.0,0.0,...,0.0,0.0,0.0,0.0,60.0,3.000000,Adware,clean,Adware,-1


Inspect the last five rows

In [18]:
print("CIC-MalMem-2022 — last 5 rows")
display(malmem.tail())

print("\nAndroid dataset — last 5 rows")
display(android.tail())

CIC-MalMem-2022 — last 5 rows


,Class,pslist.nproc,pslist.nppid,pslist.avg_threads,pslist.nprocs64bit,pslist.avg_handlers,dlllist.ndlls,dlllist.avg_dlls_per_proc,handles.nhandles,handles.avg_handles_per_proc,...,svcscan.process_services,svcscan.shared_process_services,svcscan.interactive_process_services,svcscan.nactive,callbacks.ncallbacks,callbacks.nanonymous,callbacks.ngeneric,attack_type,original_label,source_index
58591,Spyware,42,18,9.714286,0,202.738095,1596.0,38.000000,8515.0,202.738095,...,24,116,0,119.0,86.0,0,8,sample_duplication,Spyware,44788
58592,Spyware,35,15,10.514286,0,184.485714,1273.0,36.371429,6458.0,189.941177,...,21,110,0,112.0,88.0,0,8,sample_duplication,Spyware,38485
58593,Spyware,40,16,9.850000,0,213.850000,1547.0,38.675000,8555.0,219.358974,...,24,116,0,119.0,86.0,0,8,sample_duplication,Spyware,45066
58594,Spyware,40,16,9.575000,0,204.250000,1506.0,37.650000,8171.0,209.512821,...,24,116,0,119.0,87.0,0,8,sample_duplication,Spyware,43756
58595,Spyware,40,16,9.825000,0,208.675000,1557.0,38.925000,8347.0,208.675000,...,24,116,0,122.0,86.0,0,8,sample_duplication,Spyware,36374



Android dataset — last 5 rows


,ACCESS_PERSONAL_INFO___,ALTER_PHONE_STATE___,ANTI_DEBUG_____,CREATE_FOLDER_____,CREATE_PROCESS`_____,CREATE_THREAD_____,DEVICE_ACCESS_____,EXECUTE_____,FS_ACCESS____,FS_ACCESS()____,...,vibratePattern,wait4,watchRotation,windowGainedFocus,write,writev,Class,attack_type,original_label,source_index
11593,0.0,0.0,0.0,3.0,0.0,10.0,2.0,0.0,22.0,0.0,...,0.0,0.0,0.0,1.0,1162.0,10.0,Riskware,sample_duplication,Riskware,8201
11594,0.0,0.0,0.0,5.0,0.0,36.0,24.0,0.0,47.0,3.0,...,0.0,0.0,0.0,2.0,2649.0,27.0,Riskware,sample_duplication,Riskware,7696
11595,216.0,0.0,0.0,31.0,19.0,95.0,190.0,32.0,359.0,26.0,...,0.0,162.0,0.0,1.0,1731.0,992.0,Riskware,sample_duplication,Riskware,8939
11596,1.0,0.0,0.0,11.0,1.0,13.0,26.0,1.0,30.0,0.0,...,0.0,1.0,0.0,1.0,287.0,138.0,Riskware,sample_duplication,Riskware,7500
11597,4.0,0.0,0.0,1.0,0.0,14.0,3.0,0.0,22.0,1.0,...,0.0,0.0,0.0,0.0,177.0,80.0,Riskware,sample_duplication,Riskware,7822


Inspect column names

In [19]:
print("CIC-MalMem-2022 columns")
print("=" * 70)

for i, col in enumerate(malmem.columns):
    print(f"{i:>3}: {col}")

print("\n\nAndroid dataset columns")
print("=" * 70)

for i, col in enumerate(android.columns):
    print(f"{i:>3}: {col}")

CIC-MalMem-2022 columns
  0: Class
  1: pslist.nproc
  2: pslist.nppid
  3: pslist.avg_threads
  4: pslist.nprocs64bit
  5: pslist.avg_handlers
  6: dlllist.ndlls
  7: dlllist.avg_dlls_per_proc
  8: handles.nhandles
  9: handles.avg_handles_per_proc
 10: handles.nport
 11: handles.nfile
 12: handles.nevent
 13: handles.ndesktop
 14: handles.nkey
 15: handles.nthread
 16: handles.ndirectory
 17: handles.nsemaphore
 18: handles.ntimer
 19: handles.nsection
 20: handles.nmutant
 21: ldrmodules.not_in_load
 22: ldrmodules.not_in_init
 23: ldrmodules.not_in_mem
 24: ldrmodules.not_in_load_avg
 25: ldrmodules.not_in_init_avg
 26: ldrmodules.not_in_mem_avg
 27: malfind.ninjections
 28: malfind.commitCharge
 29: malfind.protection
 30: malfind.uniqueInjections
 31: psxview.not_in_pslist
 32: psxview.not_in_eprocess_pool
 33: psxview.not_in_ethread_pool
 34: psxview.not_in_pspcid_list
 35: psxview.not_in_csrss_handles
 36: psxview.not_in_session
 37: psxview.not_in_deskthrd
 38: psxview.not_in_

Verify duplicate column names

In [20]:
def duplicate_column_names(df):
    return df.columns[df.columns.duplicated()].tolist()


malmem_duplicate_columns = duplicate_column_names(malmem)
android_duplicate_columns = duplicate_column_names(android)

print("MalMem duplicate column names:")
print(malmem_duplicate_columns)

print("\nAndroid duplicate column names:")
print(android_duplicate_columns)

MalMem duplicate column names:
[]

Android duplicate column names:
[]


Check column-name whitespace

In [21]:
def columns_with_whitespace(df):
    return [
        col for col in df.columns
        if col != col.strip()
    ]


print("MalMem columns with leading/trailing whitespace:")
print(columns_with_whitespace(malmem))

print("\nAndroid columns with leading/trailing whitespace:")
print(columns_with_whitespace(android))

MalMem columns with leading/trailing whitespace:
[]

Android columns with leading/trailing whitespace:
[]


Inspect data types

In [22]:
print("MalMem data types")
print("=" * 50)
print(malmem.dtypes.value_counts())

print("\nAndroid data types")
print("=" * 50)
print(android.dtypes.value_counts())

MalMem data types
int64      33
float64    23
str         3
Name: count, dtype: int64

Android data types
float64    470
str          3
int64        1
Name: count, dtype: int64


Explicitly identify numeric and non-numeric columns

In [23]:
def inspect_column_types(df, dataset_name):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    non_numeric_cols = df.select_dtypes(exclude=np.number).columns.tolist()

    print(f"\n{dataset_name}")
    print("=" * 70)

    print(f"Total columns: {len(df.columns)}")
    print(f"Numeric columns: {len(numeric_cols)}")
    print(f"Non-numeric columns: {len(non_numeric_cols)}")

    print("\nNon-numeric columns:")
    for col in non_numeric_cols:
        print(f"  - {col}")


inspect_column_types(malmem, "CIC-MalMem-2022")
inspect_column_types(android, "Supplied Android Dataset")


CIC-MalMem-2022
Total columns: 59
Numeric columns: 56
Non-numeric columns: 3

Non-numeric columns:
  - Class
  - attack_type
  - original_label

Supplied Android Dataset
Total columns: 474
Numeric columns: 471
Non-numeric columns: 3

Non-numeric columns:
  - Class
  - attack_type
  - original_label


Verify the four critical metadata fields

In [24]:
metadata_cols = [
    "Class",
    "attack_type",
    "original_label",
    "source_index",
]

for name, df in {
    "MalMem": malmem,
    "Android": android
}.items():

    print(f"\n{name}")
    print("=" * 70)

    for col in metadata_cols:
        if col in df.columns:
            print(f"[FOUND]   {col:<20} dtype={df[col].dtype}")
        else:
            print(f"[MISSING] {col}")


MalMem
[FOUND]   Class                dtype=str
[FOUND]   attack_type          dtype=str
[FOUND]   original_label       dtype=str
[FOUND]   source_index         dtype=int64

Android
[FOUND]   Class                dtype=str
[FOUND]   attack_type          dtype=str
[FOUND]   original_label       dtype=str
[FOUND]   source_index         dtype=int64


Create a structural audit table

In [25]:
structural_audit = pd.DataFrame([
    {
        "dataset": "CIC-MalMem-2022",
        "rows": malmem.shape[0],
        "columns": malmem.shape[1],
        "numeric_columns": malmem.select_dtypes(include=np.number).shape[1],
        "non_numeric_columns": malmem.select_dtypes(exclude=np.number).shape[1],
        "has_Class": "Class" in malmem.columns,
        "has_attack_type": "attack_type" in malmem.columns,
        "has_original_label": "original_label" in malmem.columns,
        "has_source_index": "source_index" in malmem.columns,
    },
    {
        "dataset": "Supplied Android Dataset",
        "rows": android.shape[0],
        "columns": android.shape[1],
        "numeric_columns": android.select_dtypes(include=np.number).shape[1],
        "non_numeric_columns": android.select_dtypes(exclude=np.number).shape[1],
        "has_Class": "Class" in android.columns,
        "has_attack_type": "attack_type" in android.columns,
        "has_original_label": "original_label" in android.columns,
        "has_source_index": "source_index" in android.columns,
    }
])

display(structural_audit)

,dataset,rows,columns,numeric_columns,non_numeric_columns,has_Class,has_attack_type,has_original_label,has_source_index
0,CIC-MalMem-2022,58596,59,56,3,True,True,True,True
1,Supplied Android Dataset,11598,474,471,3,True,True,True,True


In [26]:
structural_audit_path = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "table_structural_audit.csv"
)

structural_audit.to_csv(structural_audit_path, index=False)

print(f"Saved:\n{structural_audit_path}")

Saved:
D:\User\Desktop\PoisonShield\results\tables\table_structural_audit.csv


Construct poisoned and Validate Attack Composition

Inspect attack types

In [27]:
print("CIC-MalMem-2022 attack types")
print("=" * 70)
print(malmem["attack_type"].value_counts(dropna=False))

print("\n\nAndroid attack types")
print("=" * 70)
print(android["attack_type"].value_counts(dropna=False))

CIC-MalMem-2022 attack types
attack_type
clean                       42202
backdoor_injection           2342
gaussian_noise_injection     2342
label_flipping               2342
feature_poisoning            2342
missing_value_injection      2342
outlier_injection            2342
sample_duplication           2342
Name: count, dtype: int64


Android attack types
attack_type
clean                       6117
backdoor_injection           783
feature_poisoning            783
missing_value_injection      783
gaussian_noise_injection     783
label_flipping               783
outlier_injection            783
sample_duplication           783
Name: count, dtype: int64


Check missing attack labels

In [28]:
for name, df in {
    "MalMem": malmem,
    "Android": android
}.items():

    missing_attack_type = df["attack_type"].isna().sum()

    print(
        f"{name}: missing attack_type values = "
        f"{missing_attack_type}"
    )

MalMem: missing attack_type values = 0
Android: missing attack_type values = 0


Create the poisoned evaluation variable

In [29]:
malmem["poisoned"] = (
    malmem["attack_type"] != "clean"
).astype(int)

android["poisoned"] = (
    android["attack_type"] != "clean"
).astype(int)

C:\Users\alokr\AppData\Local\Temp\ipykernel_15052\3305891863.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  android["poisoned"] = (


Verify the new variable

In [30]:
for name, df in {
    "MalMem": malmem,
    "Android": android
}.items():

    print(f"\n{name}")
    print("=" * 70)

    print("Unique poisoned values:")
    print(sorted(df["poisoned"].unique()))

    print("\nCounts:")
    print(df["poisoned"].value_counts().sort_index())


MalMem
Unique poisoned values:
[np.int64(0), np.int64(1)]

Counts:
poisoned
0    42202
1    16394
Name: count, dtype: int64

Android
Unique poisoned values:
[np.int64(0), np.int64(1)]

Counts:
poisoned
0    6117
1    5481
Name: count, dtype: int64


Validate the definition mathematically

In [31]:
for name, df in {
    "MalMem": malmem,
    "Android": android
}.items():

    expected_poisoned = (
        df["attack_type"] != "clean"
    ).astype(int)

    mismatches = (
        df["poisoned"] != expected_poisoned
    ).sum()

    print(
        f"{name}: poisoned-definition mismatches = {mismatches}"
    )

MalMem: poisoned-definition mismatches = 0
Android: poisoned-definition mismatches = 0


Calculate poisoning proportion

In [32]:
for name, df in {
    "MalMem": malmem,
    "Android": android
}.items():

    poisoned_rate = df["poisoned"].mean() * 100

    print(
        f"{name}: {poisoned_rate:.3f}% of observations are poisoned"
    )

MalMem: 27.978% of observations are poisoned
Android: 47.258% of observations are poisoned


Build a complete attack-distribution table

In [33]:
attack_distribution = pd.concat(
    [
        (
            malmem["attack_type"]
            .value_counts()
            .rename_axis("attack_type")
            .reset_index(name="count")
            .assign(dataset="CIC-MalMem-2022")
        ),
        (
            android["attack_type"]
            .value_counts()
            .rename_axis("attack_type")
            .reset_index(name="count")
            .assign(dataset="Android")
        ),
    ],
    ignore_index=True
)

attack_distribution["percentage"] = (
    attack_distribution.groupby("dataset")["count"]
    .transform(lambda x: 100 * x / x.sum())
)

attack_distribution = attack_distribution[
    ["dataset", "attack_type", "count", "percentage"]
].sort_values(
    ["dataset", "attack_type"]
).reset_index(drop=True)

display(attack_distribution)

,dataset,attack_type,count,percentage
0,Android,backdoor_injection,783,6.751164
1,Android,clean,6117,52.741852
2,Android,feature_poisoning,783,6.751164
3,Android,gaussian_noise_injection,783,6.751164
4,Android,label_flipping,783,6.751164
5,Android,missing_value_injection,783,6.751164
6,Android,outlier_injection,783,6.751164
7,Android,sample_duplication,783,6.751164
8,CIC-MalMem-2022,backdoor_injection,2342,3.996860
9,CIC-MalMem-2022,clean,42202,72.021981


Add poisoned counts

In [34]:
poisoned_summary = []

for name, df in {
    "CIC-MalMem-2022": malmem,
    "Android": android
}.items():

    poisoned_count = int(df["poisoned"].sum())
    clean_count = int((df["poisoned"] == 0).sum())
    total = len(df)

    poisoned_summary.append({
        "dataset": name,
        "total_observations": total,
        "clean_observations": clean_count,
        "poisoned_observations": poisoned_count,
        "clean_percentage": 100 * clean_count / total,
        "poisoned_percentage": 100 * poisoned_count / total
    })

poisoned_summary = pd.DataFrame(poisoned_summary)

display(poisoned_summary)

,dataset,total_observations,clean_observations,poisoned_observations,clean_percentage,poisoned_percentage
0,CIC-MalMem-2022,58596,42202,16394,72.021981,27.978019
1,Android,11598,6117,5481,52.741852,47.258148


Save the audit artifacts

In [35]:
attack_table_path = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "table_attack_distribution.csv"
)

poisoned_summary_path = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "table_poisoned_summary.csv"
)

attack_distribution.to_csv(
    attack_table_path,
    index=False
)

poisoned_summary.to_csv(
    poisoned_summary_path,
    index=False
)

print("Saved:")
print(attack_table_path)
print(poisoned_summary_path)

Saved:
D:\User\Desktop\PoisonShield\results\tables\table_attack_distribution.csv
D:\User\Desktop\PoisonShield\results\tables\table_poisoned_summary.csv


Create a validation report

In [36]:
validation_checks = []

for name, df in {
    "CIC-MalMem-2022": malmem,
    "Android": android
}.items():

    attack_types = set(df["attack_type"].dropna().unique())

    expected_attack_types = {
        "clean",
        "backdoor_injection",
        "feature_poisoning",
        "gaussian_noise_injection",
        "label_flipping",
        "missing_value_injection",
        "sample_duplication",
        "outlier_injection"
    }

    validation_checks.append({
        "dataset": name,
        "attack_type_count": len(attack_types),
        "attack_types_correct": attack_types == expected_attack_types,
        "missing_attack_type": int(df["attack_type"].isna().sum()),
        "poisoned_values_valid": set(df["poisoned"].unique()) == {0, 1},
        "poisoned_definition_valid": (
            df["poisoned"]
            == (df["attack_type"] != "clean").astype(int)
        ).all()
    })

validation_report = pd.DataFrame(validation_checks)

display(validation_report)

,dataset,attack_type_count,attack_types_correct,missing_attack_type,poisoned_values_valid,poisoned_definition_valid
0,CIC-MalMem-2022,8,True,0,True,True
1,Android,8,True,0,True,True


Label Consistency Audit

Overall label mismatch count

In [37]:
for name, df in {
    "CIC-MalMem-2022": malmem,
    "Android": android
}.items():

    mismatch_mask = df["Class"] != df["original_label"]

    mismatch_count = mismatch_mask.sum()
    mismatch_percentage = 100 * mismatch_count / len(df)

    print(f"\n{name}")
    print("=" * 70)
    print(f"Total observations: {len(df):,}")
    print(f"Label mismatches:   {mismatch_count:,}")
    print(f"Mismatch rate:      {mismatch_percentage:.3f}%")


CIC-MalMem-2022
Total observations: 58,596
Label mismatches:   4,684
Mismatch rate:      7.994%

Android
Total observations: 11,598
Label mismatches:   1,566
Mismatch rate:      13.502%


Verify expected mismatch counts

In [38]:
expected_mismatches = {
    "CIC-MalMem-2022": 4684,
    "Android": 1566
}

actual_datasets = {
    "CIC-MalMem-2022": malmem,
    "Android": android
}

for name, df in actual_datasets.items():

    actual = int(
        (df["Class"] != df["original_label"]).sum()
    )

    expected = expected_mismatches[name]

    status = "PASS" if actual == expected else "FAIL"

    print(
        f"{name}: {status} | "
        f"Expected={expected:,}, Actual={actual:,}"
    )

CIC-MalMem-2022: PASS | Expected=4,684, Actual=4,684
Android: PASS | Expected=1,566, Actual=1,566


Mismatches by attack type

In [39]:
for name, df in actual_datasets.items():

    mismatch_mask = (
        df["Class"] != df["original_label"]
    )

    mismatch_by_attack = (
        df.loc[mismatch_mask, "attack_type"]
        .value_counts()
        .sort_index()
    )

    print(f"\n{name}")
    print("=" * 70)
    print(mismatch_by_attack)


CIC-MalMem-2022
attack_type
backdoor_injection    2342
label_flipping        2342
Name: count, dtype: int64

Android
attack_type
backdoor_injection    783
label_flipping        783
Name: count, dtype: int64


Build the attack × mismatch contingency table

In [40]:
for name, df in actual_datasets.items():

    mismatch = (
        df["Class"] != df["original_label"]
    )

    table = pd.crosstab(
        df["attack_type"],
        mismatch,
        margins=True
    )

    table.columns = [
        "Match" if col is False
        else "Mismatch" if col is True
        else col
        for col in table.columns
    ]

    print(f"\n{name}")
    print("=" * 70)
    display(table)


CIC-MalMem-2022


,Match,Mismatch,All
attack_type,,,
backdoor_injection,0,2342,2342
clean,42202,0,42202
feature_poisoning,2342,0,2342
gaussian_noise_injection,2342,0,2342
label_flipping,0,2342,2342
missing_value_injection,2342,0,2342
outlier_injection,2342,0,2342
sample_duplication,2342,0,2342
All,53912,4684,58596



Android


,Match,Mismatch,All
attack_type,,,
backdoor_injection,0,783,783
clean,6117,0,6117
feature_poisoning,783,0,783
gaussian_noise_injection,783,0,783
label_flipping,0,783,783
missing_value_injection,783,0,783
outlier_injection,783,0,783
sample_duplication,783,0,783
All,10032,1566,11598


Check whether clean observations have label mismatches

In [41]:
for name, df in actual_datasets.items():

    clean_df = df[df["attack_type"] == "clean"]

    clean_mismatches = (
        clean_df["Class"] != clean_df["original_label"]
    ).sum()

    print(
        f"{name}: clean observations with label mismatch = "
        f"{clean_mismatches}"
    )

CIC-MalMem-2022: clean observations with label mismatch = 0
Android: clean observations with label mismatch = 0


Inspect the actual label transitions

In [42]:
for name, df in actual_datasets.items():

    mismatch_df = df[
        df["Class"] != df["original_label"]
    ]

    transitions = pd.crosstab(
        mismatch_df["original_label"],
        mismatch_df["Class"]
    )

    print(f"\n{name} — Label transitions")
    print("=" * 70)
    display(transitions)


CIC-MalMem-2022 — Label transitions


Class,Benign
original_label,
Ransomware,1566
Spyware,1602
Trojan,1516



Android — Label transitions


Class,Benign
original_label,
Adware,200
Banking,336
Riskware,406
SMS malware,624


Inspect transitions separately by attack

In [43]:
for name, df in actual_datasets.items():

    print(f"\n{'=' * 70}")
    print(name)
    print(f"{'=' * 70}")

    mismatch_df = df[
        df["Class"] != df["original_label"]
    ]

    for attack in sorted(
        mismatch_df["attack_type"].unique()
    ):

        attack_df = mismatch_df[
            mismatch_df["attack_type"] == attack
        ]

        transitions = pd.crosstab(
            attack_df["original_label"],
            attack_df["Class"]
        )

        print(f"\nAttack: {attack}")
        display(transitions)


CIC-MalMem-2022

Attack: backdoor_injection


Class,Benign
original_label,
Ransomware,783
Spyware,801
Trojan,758



Attack: label_flipping


Class,Benign
original_label,
Ransomware,783
Spyware,801
Trojan,758



Android

Attack: backdoor_injection


Class,Benign
original_label,
Adware,100
Banking,168
Riskware,203
SMS malware,312



Attack: label_flipping


Class,Benign
original_label,
Adware,100
Banking,168
Riskware,203
SMS malware,312


Verify source-index behavior

In [44]:
for name, df in actual_datasets.items():

    print(f"\n{name}")
    print("=" * 70)

    print(
        df["source_index"]
        .describe()
    )

    print("\nUnique source_index values:")
    print(df["source_index"].nunique())


CIC-MalMem-2022
count    58596.000000
mean      1761.379582
std       8801.247015
min         -1.000000
25%         -1.000000
50%         -1.000000
75%         -1.000000
max      58562.000000
Name: source_index, dtype: float64

Unique source_index values:
2343

Android
count    11598.000000
mean       326.775306
std       1417.417923
min         -1.000000
25%         -1.000000
50%         -1.000000
75%         -1.000000
max       9788.000000
Name: source_index, dtype: float64

Unique source_index values:
784


Check source_index by attack

In [45]:
for name, df in actual_datasets.items():

    source_summary = (
        df.groupby("attack_type")["source_index"]
        .agg(
            count="count",
            unique_values="nunique",
            minimum="min",
            maximum="max",
            non_negative=lambda x: (x >= 0).sum()
        )
    )

    print(f"\n{name}")
    print("=" * 70)
    display(source_summary)


CIC-MalMem-2022


,count,unique_values,minimum,maximum,non_negative
attack_type,,,,,
backdoor_injection,2342,1,-1,-1,0
clean,42202,1,-1,-1,0
feature_poisoning,2342,1,-1,-1,0
gaussian_noise_injection,2342,1,-1,-1,0
label_flipping,2342,1,-1,-1,0
missing_value_injection,2342,1,-1,-1,0
outlier_injection,2342,1,-1,-1,0
sample_duplication,2342,2342,29307,58562,2342



Android


,count,unique_values,minimum,maximum,non_negative
attack_type,,,,,
backdoor_injection,783,1,-1,-1,0
clean,6117,1,-1,-1,0
feature_poisoning,783,1,-1,-1,0
gaussian_noise_injection,783,1,-1,-1,0
label_flipping,783,1,-1,-1,0
missing_value_injection,783,1,-1,-1,0
outlier_injection,783,1,-1,-1,0
sample_duplication,783,783,4,9788,783


Verify duplication source-index coverage

In [46]:
for name, df in actual_datasets.items():

    duplication_df = df[
        df["attack_type"] == "sample_duplication"
    ]

    valid_source_index = (
        duplication_df["source_index"] >= 0
    ).sum()

    print(
        f"{name}: "
        f"{valid_source_index:,} / "
        f"{len(duplication_df):,} duplication rows "
        f"have valid source_index"
    )

CIC-MalMem-2022: 2,342 / 2,342 duplication rows have valid source_index
Android: 783 / 783 duplication rows have valid source_index


Save the label audit

In [47]:
label_audit_rows = []

for name, df in actual_datasets.items():

    mismatch_mask = (
        df["Class"] != df["original_label"]
    )

    for attack in sorted(df["attack_type"].unique()):

        attack_df = df[
            df["attack_type"] == attack
        ]

        attack_mismatch = (
            attack_df["Class"]
            != attack_df["original_label"]
        )

        label_audit_rows.append({
            "dataset": name,
            "attack_type": attack,
            "observations": len(attack_df),
            "label_mismatches": int(attack_mismatch.sum()),
            "label_match": int((~attack_mismatch).sum()),
            "mismatch_percentage": (
                100 * attack_mismatch.mean()
            )
        })

label_audit = pd.DataFrame(label_audit_rows)

display(label_audit)

,dataset,attack_type,observations,label_mismatches,label_match,mismatch_percentage
0,CIC-MalMem-2022,backdoor_injection,2342,2342,0,100.0
1,CIC-MalMem-2022,clean,42202,0,42202,0.0
2,CIC-MalMem-2022,feature_poisoning,2342,0,2342,0.0
3,CIC-MalMem-2022,gaussian_noise_injection,2342,0,2342,0.0
4,CIC-MalMem-2022,label_flipping,2342,2342,0,100.0
5,CIC-MalMem-2022,missing_value_injection,2342,0,2342,0.0
6,CIC-MalMem-2022,outlier_injection,2342,0,2342,0.0
7,CIC-MalMem-2022,sample_duplication,2342,0,2342,0.0
8,Android,backdoor_injection,783,783,0,100.0
9,Android,clean,6117,0,6117,0.0


In [48]:
label_audit_path = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "table_label_consistency_audit.csv"
)

label_audit.to_csv(
    label_audit_path,
    index=False
)

print(f"Saved:\n{label_audit_path}")

Saved:
D:\User\Desktop\PoisonShield\results\tables\table_label_consistency_audit.csv


Create the final checkpoint

In [49]:
label_validation = []

for name, df in actual_datasets.items():

    mismatch_count = int(
        (df["Class"] != df["original_label"]).sum()
    )

    clean_mismatch = int(
        (
            df.loc[
                df["attack_type"] == "clean",
                "Class"
            ]
            !=
            df.loc[
                df["attack_type"] == "clean",
                "original_label"
            ]
        ).sum()
    )

    duplication = df[
        df["attack_type"] == "sample_duplication"
    ]

    valid_dup_source = int(
        (duplication["source_index"] >= 0).sum()
    )

    non_duplication = df[
        df["attack_type"] != "sample_duplication"
    ]

    unexpected_source = int(
        (non_duplication["source_index"] >= 0).sum()
    )

    expected = expected_mismatches[name]

    label_validation.append({
        "dataset": name,
        "mismatch_count": mismatch_count,
        "expected_mismatch_count": expected,
        "mismatch_count_pass": mismatch_count == expected,
        "clean_mismatch_count": clean_mismatch,
        "clean_mismatch_pass": clean_mismatch == 0,
        "valid_duplication_source_indices": valid_dup_source,
        "duplication_source_index_pass": (
            valid_dup_source == len(duplication)
        ),
        "unexpected_non_duplication_source_indices": unexpected_source,
        "non_duplication_source_index_pass": (
            unexpected_source == 0
        )
    })

label_validation = pd.DataFrame(label_validation)

display(label_validation)

,dataset,mismatch_count,expected_mismatch_count,mismatch_count_pass,clean_mismatch_count,clean_mismatch_pass,valid_duplication_source_indices,duplication_source_index_pass,unexpected_non_duplication_source_indices,non_duplication_source_index_pass
0,CIC-MalMem-2022,4684,4684,True,0,True,2342,True,0,True
1,Android,1566,1566,True,0,True,783,True,0,True


Missing-Value Audit

Overall missing-value counts

In [50]:
# Phase 2.5.1 — Overall missing-value audit

malmem_missing_cells = int(malmem.isna().sum().sum())
android_missing_cells = int(android.isna().sum().sum())

malmem_missing_rows = int(malmem.isna().any(axis=1).sum())
android_missing_rows = int(android.isna().any(axis=1).sum())

print("CIC-MalMem-2022")
print(f"  Missing cells: {malmem_missing_cells:,}")
print(f"  Rows with >=1 missing value: {malmem_missing_rows:,}")

print("\nAndroid")
print(f"  Missing cells: {android_missing_cells:,}")
print(f"  Rows with >=1 missing value: {android_missing_rows:,}")

CIC-MalMem-2022
  Missing cells: 4,684
  Rows with >=1 missing value: 2,342

Android
  Missing cells: 1,566
  Rows with >=1 missing value: 783


Missing values by column

In [51]:
# Phase 2.5.2 — Missing values by feature/column

malmem_missing_by_column = (
    malmem.isna()
    .sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
)

android_missing_by_column = (
    android.isna()
    .sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
)

print("CIC-MalMem-2022 — Missing values by column")
print(malmem_missing_by_column)

print("\nAndroid — Missing values by column")
print(android_missing_by_column)

CIC-MalMem-2022 — Missing values by column
svcscan.kernel_drivers    970
malfind.commitCharge      941
dlllist.ndlls             933
pslist.avg_handlers       921
handles.nhandles          919
dtype: int64

Android — Missing values by column
clock_gettime    360
writev           335
sched_yield      322
gettimeofday     282
read             267
dtype: int64


Missing values by attack type

In [52]:
# Phase 2.5.3 — Missing values by attack type

malmem_missing_by_attack = (
    malmem.assign(has_missing=malmem.isna().any(axis=1))
    .groupby("attack_type")["has_missing"]
    .sum()
    .astype(int)
)

android_missing_by_attack = (
    android.assign(has_missing=android.isna().any(axis=1))
    .groupby("attack_type")["has_missing"]
    .sum()
    .astype(int)
)

print("CIC-MalMem-2022 — Rows with missing values by attack_type")
print(malmem_missing_by_attack)

print("\nAndroid — Rows with missing values by attack_type")
print(android_missing_by_attack)

CIC-MalMem-2022 — Rows with missing values by attack_type
attack_type
backdoor_injection             0
clean                          0
feature_poisoning              0
gaussian_noise_injection       0
label_flipping                 0
missing_value_injection     2342
outlier_injection              0
sample_duplication             0
Name: has_missing, dtype: int64

Android — Rows with missing values by attack_type
attack_type
backdoor_injection            0
clean                         0
feature_poisoning             0
gaussian_noise_injection      0
label_flipping                0
missing_value_injection     783
outlier_injection             0
sample_duplication            0
Name: has_missing, dtype: int64


C:\Users\alokr\AppData\Local\Temp\ipykernel_15052\3600159418.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  android.assign(has_missing=android.isna().any(axis=1))


Missing cells by attack type

In [53]:
# Phase 2.5.4 — Missing cells by attack type

def missing_cells_by_attack(df):
    return (
        df.groupby("attack_type")
        .apply(lambda x: int(x.isna().sum().sum()), include_groups=False)
        .sort_values(ascending=False)
    )

malmem_missing_cells_by_attack = missing_cells_by_attack(malmem)
android_missing_cells_by_attack = missing_cells_by_attack(android)

print("CIC-MalMem-2022 — Missing cells by attack_type")
print(malmem_missing_cells_by_attack)

print("\nAndroid — Missing cells by attack_type")
print(android_missing_cells_by_attack)

CIC-MalMem-2022 — Missing cells by attack_type
attack_type
missing_value_injection     4684
backdoor_injection             0
feature_poisoning              0
clean                          0
gaussian_noise_injection       0
label_flipping                 0
outlier_injection              0
sample_duplication             0
dtype: int64

Android — Missing cells by attack_type
attack_type
missing_value_injection     1566
backdoor_injection             0
feature_poisoning              0
clean                          0
gaussian_noise_injection       0
label_flipping                 0
outlier_injection              0
sample_duplication             0
dtype: int64


Missing cells per affected row

In [54]:
# Phase 2.5.5 — Missing cells per affected row

malmem_missing_per_row = malmem.isna().sum(axis=1)
android_missing_per_row = android.isna().sum(axis=1)

print("CIC-MalMem-2022 — Missing cells per affected row")
print(
    malmem_missing_per_row[
        malmem_missing_per_row > 0
    ].value_counts().sort_index()
)

print("\nAndroid — Missing cells per affected row")
print(
    android_missing_per_row[
        android_missing_per_row > 0
    ].value_counts().sort_index()
)

CIC-MalMem-2022 — Missing cells per affected row
2    2342
Name: count, dtype: int64

Android — Missing cells per affected row
2    783
Name: count, dtype: int64


In [55]:
# Phase 2.5.6 — Verify missingness does not occur
# in clean or non-missing-value-injection attacks

for dataset_name, df in {
    "CIC-MalMem-2022": malmem,
    "Android": android
}.items():

    row_has_missing = df.isna().any(axis=1)

    clean_missing_rows = int(
        row_has_missing[
            df["attack_type"] == "clean"
        ].sum()
    )

    non_target_missing_rows = int(
        row_has_missing[
            ~df["attack_type"].isin(
                ["clean", "missing_value_injection"]
            )
        ].sum()
    )

    target_missing_rows = int(
        row_has_missing[
            df["attack_type"] == "missing_value_injection"
        ].sum()
    )

    print(f"\n{dataset_name}")
    print(f"Clean rows with missing values: {clean_missing_rows}")
    print(
        "Non-target attack rows with missing values: "
        f"{non_target_missing_rows}"
    )
    print(
        "Missing-value-injection rows with missing values: "
        f"{target_missing_rows}"
    )


CIC-MalMem-2022
Clean rows with missing values: 0
Non-target attack rows with missing values: 0
Missing-value-injection rows with missing values: 2342

Android
Clean rows with missing values: 0
Non-target attack rows with missing values: 0
Missing-value-injection rows with missing values: 783


In [56]:
# Phase 2.5.7 — Save missing-value audit tables

results_dir = Path("../results/tables")
results_dir.mkdir(parents=True, exist_ok=True)

# Overall missingness
missing_overall = pd.DataFrame([
    {
        "dataset": "CIC-MalMem-2022",
        "missing_cells": int(malmem.isna().sum().sum()),
        "rows_with_missing": int(malmem.isna().any(axis=1).sum())
    },
    {
        "dataset": "Android",
        "missing_cells": int(android.isna().sum().sum()),
        "rows_with_missing": int(android.isna().any(axis=1).sum())
    }
])

# Missingness by feature
missing_by_feature = pd.concat([
    malmem_missing_by_column.rename("missing_count")
    .rename_axis("feature")
    .reset_index()
    .assign(dataset="CIC-MalMem-2022"),

    android_missing_by_column.rename("missing_count")
    .rename_axis("feature")
    .reset_index()
    .assign(dataset="Android")
], ignore_index=True)

missing_by_feature = missing_by_feature[
    ["dataset", "feature", "missing_count"]
]

# Missingness by attack
missing_by_attack = pd.concat([
    malmem_missing_by_attack.rename("rows_with_missing")
    .rename_axis("attack_type")
    .reset_index()
    .assign(dataset="CIC-MalMem-2022"),

    android_missing_by_attack.rename("rows_with_missing")
    .rename_axis("attack_type")
    .reset_index()
    .assign(dataset="Android")
], ignore_index=True)

missing_by_attack = missing_by_attack[
    ["dataset", "attack_type", "rows_with_missing"]
]

# Missing cells by attack
missing_cells_attack = pd.concat([
    malmem_missing_cells_by_attack.rename("missing_cells")
    .rename_axis("attack_type")
    .reset_index()
    .assign(dataset="CIC-MalMem-2022"),

    android_missing_cells_by_attack.rename("missing_cells")
    .rename_axis("attack_type")
    .reset_index()
    .assign(dataset="Android")
], ignore_index=True)

missing_cells_attack = missing_cells_attack[
    ["dataset", "attack_type", "missing_cells"]
]

# Save
missing_overall.to_csv(
    results_dir / "table_missingness_overall.csv",
    index=False
)

missing_by_feature.to_csv(
    results_dir / "table_missingness_by_feature.csv",
    index=False
)

missing_by_attack.to_csv(
    results_dir / "table_missingness_by_attack.csv",
    index=False
)

missing_cells_attack.to_csv(
    results_dir / "table_missing_cells_by_attack.csv",
    index=False
)

print("Saved Phase 2.5 audit tables:")
print(" - table_missingness_overall.csv")
print(" - table_missingness_by_feature.csv")
print(" - table_missingness_by_attack.csv")
print(" - table_missing_cells_by_attack.csv")

Saved Phase 2.5 audit tables:
 - table_missingness_overall.csv
 - table_missingness_by_feature.csv
 - table_missingness_by_attack.csv
 - table_missing_cells_by_attack.csv


Duplicate Audit

In [57]:
# Phase 2.6.1 — Overall exact duplicate count

malmem_exact_duplicates = int(malmem.duplicated().sum())
android_exact_duplicates = int(android.duplicated().sum())

print("CIC-MalMem-2022 exact duplicate rows:", malmem_exact_duplicates)
print("Android exact duplicate rows:", android_exact_duplicates)

CIC-MalMem-2022 exact duplicate rows: 170
Android exact duplicate rows: 33


In [58]:
# Phase 2.6.2 — Duplicate rows by attack type

def duplicate_rows_by_attack(df):
    duplicate_mask = df.duplicated(keep=False)

    return (
        df.loc[duplicate_mask]
        .groupby("attack_type")
        .size()
        .sort_values(ascending=False)
    )

malmem_duplicates_by_attack = duplicate_rows_by_attack(malmem)
android_duplicates_by_attack = duplicate_rows_by_attack(android)

print("CIC-MalMem-2022 — duplicate rows by attack_type")
print(malmem_duplicates_by_attack)

print("\nAndroid — duplicate rows by attack_type")
print(android_duplicates_by_attack)

CIC-MalMem-2022 — duplicate rows by attack_type
attack_type
clean                 277
backdoor_injection      8
label_flipping          6
dtype: int64

Android — duplicate rows by attack_type
attack_type
clean                       38
label_flipping               5
backdoor_injection           2
feature_poisoning            2
gaussian_noise_injection     2
dtype: int64


In [59]:
# Phase 2.6.3 — Exact duplicates within clean observations

for dataset_name, df in {
    "CIC-MalMem-2022": malmem,
    "Android": android
}.items():

    clean_df = df[df["attack_type"] == "clean"]

    clean_duplicates = int(clean_df.duplicated().sum())

    print(
        f"{dataset_name} — exact duplicate rows within clean data: "
        f"{clean_duplicates}"
    )

CIC-MalMem-2022 — exact duplicate rows within clean data: 163
Android — exact duplicate rows within clean data: 26


In [60]:
# Phase 2.6.4 — Duplicate structure within sample_duplication

for dataset_name, df in {
    "CIC-MalMem-2022": malmem,
    "Android": android
}.items():

    duplication_df = df[
        df["attack_type"] == "sample_duplication"
    ]

    exact_duplicates = int(
        duplication_df.duplicated().sum()
    )

    duplicate_groups = int(
        duplication_df.duplicated(
            keep=False
        ).sum()
    )

    print(f"\n{dataset_name}")
    print(
        "sample_duplication rows:",
        len(duplication_df)
    )
    print(
        "Exact duplicate rows within sample_duplication:",
        exact_duplicates
    )
    print(
        "Rows belonging to duplicate groups:",
        duplicate_groups
    )


CIC-MalMem-2022
sample_duplication rows: 2342
Exact duplicate rows within sample_duplication: 0
Rows belonging to duplicate groups: 0

Android
sample_duplication rows: 783
Exact duplicate rows within sample_duplication: 0
Rows belonging to duplicate groups: 0


In [61]:
# Phase 2.6.5 — Validate source_index in sample_duplication

for dataset_name, df in {
    "CIC-MalMem-2022": malmem,
    "Android": android
}.items():

    duplication_df = df[
        df["attack_type"] == "sample_duplication"
    ]

    valid_source_indices = duplication_df[
        duplication_df["source_index"].notna()
        & (duplication_df["source_index"] >= 0)
    ]

    invalid_source_indices = duplication_df[
        duplication_df["source_index"].isna()
        | (duplication_df["source_index"] < 0)
    ]

    print(f"\n{dataset_name}")
    print(
        "sample_duplication rows:",
        len(duplication_df)
    )
    print(
        "Valid source_index:",
        len(valid_source_indices)
    )
    print(
        "Invalid/missing source_index:",
        len(invalid_source_indices)
    )

    print(
        "source_index range:",
        (
            valid_source_indices["source_index"].min(),
            valid_source_indices["source_index"].max()
        )
    )


CIC-MalMem-2022
sample_duplication rows: 2342
Valid source_index: 2342
Invalid/missing source_index: 0
source_index range: (np.int64(29307), np.int64(58562))

Android
sample_duplication rows: 783
Valid source_index: 783
Invalid/missing source_index: 0
source_index range: (np.int64(4), np.int64(9788))


In [62]:
# Phase 2.6.6 — Check source_index mapping to clean observations

for dataset_name, df in {
    "CIC-MalMem-2022": malmem,
    "Android": android
}.items():

    duplication_df = df[
        df["attack_type"] == "sample_duplication"
    ].copy()

    clean_df = df[
        df["attack_type"] == "clean"
    ].copy()

    clean_source_indices = set(
        clean_df.loc[
            clean_df["source_index"] >= 0,
            "source_index"
        ].astype(int)
    )

    duplication_source_indices = set(
        duplication_df["source_index"].astype(int)
    )

    mapped_to_clean = (
        duplication_source_indices
        & clean_source_indices
    )

    not_mapped_to_clean = (
        duplication_source_indices
        - clean_source_indices
    )

    print(f"\n{dataset_name}")
    print(
        "Unique sample_duplication source_index values:",
        len(duplication_source_indices)
    )
    print(
        "Source indices also present in clean rows:",
        len(mapped_to_clean)
    )
    print(
        "Source indices NOT present in clean rows:",
        len(not_mapped_to_clean)
    )


CIC-MalMem-2022
Unique sample_duplication source_index values: 2342
Source indices also present in clean rows: 0
Source indices NOT present in clean rows: 2342

Android
Unique sample_duplication source_index values: 783
Source indices also present in clean rows: 0
Source indices NOT present in clean rows: 783


In [63]:
# Phase 2.6.7 — Feature-vector matching against clean observations

metadata_cols = [
    "Class",
    "attack_type",
    "original_label",
    "source_index",
    "poisoned"
]

def feature_columns(df):
    return [
        c for c in df.columns
        if c not in metadata_cols
        and pd.api.types.is_numeric_dtype(df[c])
    ]


for dataset_name, df in {
    "CIC-MalMem-2022": malmem,
    "Android": android
}.items():

    features = feature_columns(df)

    clean_df = df[
        df["attack_type"] == "clean"
    ].copy()

    duplication_df = df[
        df["attack_type"] == "sample_duplication"
    ].copy()

    # Hash the feature vectors for efficient comparison
    clean_hashes = set(
        pd.util.hash_pandas_object(
            clean_df[features],
            index=False
        )
    )

    duplication_hashes = pd.util.hash_pandas_object(
        duplication_df[features],
        index=False
    )

    matched_rows = duplication_hashes.isin(
        clean_hashes
    )

    print(f"\n{dataset_name}")
    print("Substantive feature columns:", len(features))
    print(
        "sample_duplication rows:",
        len(duplication_df)
    )
    print(
        "Rows with exact clean feature-vector match:",
        int(matched_rows.sum())
    )
    print(
        "Rows without exact clean feature-vector match:",
        int((~matched_rows).sum())
    )


CIC-MalMem-2022
Substantive feature columns: 55
sample_duplication rows: 2342
Rows with exact clean feature-vector match: 2342
Rows without exact clean feature-vector match: 0

Android
Substantive feature columns: 470
sample_duplication rows: 783
Rows with exact clean feature-vector match: 783
Rows without exact clean feature-vector match: 0


In [64]:
# Phase 2.6.8 — Clean feature-vector overlap by attack type

for dataset_name, df in {
    "CIC-MalMem-2022": malmem,
    "Android": android
}.items():

    features = feature_columns(df)

    clean_df = df[
        df["attack_type"] == "clean"
    ]

    clean_hashes = set(
        pd.util.hash_pandas_object(
            clean_df[features],
            index=False
        )
    )

    print(f"\n{dataset_name}")
    print("Exact clean feature-vector matches by attack:")

    for attack_type in sorted(
        df["attack_type"].unique()
    ):

        attack_df = df[
            df["attack_type"] == attack_type
        ]

        attack_hashes = pd.util.hash_pandas_object(
            attack_df[features],
            index=False
        )

        matched = int(
            attack_hashes.isin(clean_hashes).sum()
        )

        print(
            f"  {attack_type}: {matched}"
        )


CIC-MalMem-2022
Exact clean feature-vector matches by attack:
  backdoor_injection: 0
  clean: 42202
  feature_poisoning: 0
  gaussian_noise_injection: 0
  label_flipping: 25
  missing_value_injection: 0
  outlier_injection: 0
  sample_duplication: 2342

Android
Exact clean feature-vector matches by attack:
  backdoor_injection: 0
  clean: 6117
  feature_poisoning: 2
  gaussian_noise_injection: 4
  label_flipping: 9
  missing_value_injection: 0
  outlier_injection: 0
  sample_duplication: 783


In [65]:
# Phase 2.6.9 — Duplicate group structure

def duplicate_group_summary(df):

    duplicate_mask = df.duplicated(
        keep=False
    )

    duplicate_df = df.loc[
        duplicate_mask
    ].copy()

    if duplicate_df.empty:
        return pd.DataFrame(
            columns=[
                "duplicate_group_size",
                "number_of_groups",
                "rows_in_groups"
            ]
        )

    group_sizes = (
        duplicate_df
        .groupby(list(df.columns), dropna=False)
        .size()
    )

    summary = (
        group_sizes
        .value_counts()
        .sort_index()
        .rename_axis("duplicate_group_size")
        .rename("number_of_groups")
        .reset_index()
    )

    summary["rows_in_groups"] = (
        summary["duplicate_group_size"]
        * summary["number_of_groups"]
    )

    return summary


for dataset_name, df in {
    "CIC-MalMem-2022": malmem,
    "Android": android
}.items():

    summary = duplicate_group_summary(df)

    print(f"\n{dataset_name} — duplicate group structure")
    print(summary)


CIC-MalMem-2022 — duplicate group structure
   duplicate_group_size  number_of_groups  rows_in_groups
0                     2               108             216
1                     3                 2               6
2                     4                 5              20
3                     6                 2              12
4                     8                 2              16
5                    10                 1              10
6                    11                 1              11

Android — duplicate group structure
   duplicate_group_size  number_of_groups  rows_in_groups
0                     2                10              20
1                     3                 2               6
2                     4                 1               4
3                     5                 1               5
4                     7                 2              14


In [66]:
# Phase 2.6.10 — Save duplicate audit artifacts

results_dir = Path("../results/tables")
results_dir.mkdir(parents=True, exist_ok=True)

# Overall
duplicate_overall = pd.DataFrame([
    {
        "dataset": "CIC-MalMem-2022",
        "exact_duplicate_rows": malmem_exact_duplicates
    },
    {
        "dataset": "Android",
        "exact_duplicate_rows": android_exact_duplicates
    }
])

# By attack
duplicate_attack = pd.concat([
    malmem_duplicates_by_attack.rename(
        "duplicate_rows"
    ).rename_axis("attack_type")
    .reset_index()
    .assign(dataset="CIC-MalMem-2022"),

    android_duplicates_by_attack.rename(
        "duplicate_rows"
    ).rename_axis("attack_type")
    .reset_index()
    .assign(dataset="Android")
], ignore_index=True)

duplicate_attack = duplicate_attack[
    ["dataset", "attack_type", "duplicate_rows"]
]

duplicate_overall.to_csv(
    results_dir / "table_duplicate_overall.csv",
    index=False
)

duplicate_attack.to_csv(
    results_dir / "table_duplicate_by_attack.csv",
    index=False
)

print("Saved Phase 2.6 duplicate audit tables:")
print(" - table_duplicate_overall.csv")
print(" - table_duplicate_by_attack.csv")

Saved Phase 2.6 duplicate audit tables:
 - table_duplicate_overall.csv
 - table_duplicate_by_attack.csv


In [67]:
# Phase 2.6.11 — Final duplicate audit validation

validation_rows = []

for dataset_name, df in {
    "CIC-MalMem-2022": malmem,
    "Android": android
}.items():

    if dataset_name == "CIC-MalMem-2022":
        expected_duplicates = 170
        expected_clean_duplicates = 163
        expected_duplication_rows = 2342
    else:
        expected_duplicates = 33
        expected_clean_duplicates = 26
        expected_duplication_rows = 783

    exact_duplicates = int(
        df.duplicated().sum()
    )

    clean_df = df[
        df["attack_type"] == "clean"
    ]

    clean_duplicates = int(
        clean_df.duplicated().sum()
    )

    duplication_df = df[
        df["attack_type"] == "sample_duplication"
    ]

    valid_source_index = int(
        (
            duplication_df["source_index"].notna()
            & (duplication_df["source_index"] >= 0)
        ).sum()
    )

    invalid_source_index = int(
        len(duplication_df) - valid_source_index
    )

    # Feature-vector match against clean
    features = feature_columns(df)

    clean_hashes = set(
        pd.util.hash_pandas_object(
            clean_df[features],
            index=False
        )
    )

    duplication_hashes = pd.util.hash_pandas_object(
        duplication_df[features],
        index=False
    )

    feature_matches = int(
        duplication_hashes.isin(
            clean_hashes
        ).sum()
    )

    validation_rows.append({
        "dataset": dataset_name,

        "exact_duplicate_rows": exact_duplicates,
        "expected_exact_duplicate_rows": expected_duplicates,
        "exact_duplicate_count_pass": (
            exact_duplicates == expected_duplicates
        ),

        "clean_duplicate_rows": clean_duplicates,
        "expected_clean_duplicate_rows": expected_clean_duplicates,
        "clean_duplicate_count_pass": (
            clean_duplicates == expected_clean_duplicates
        ),

        "sample_duplication_rows": len(duplication_df),
        "expected_sample_duplication_rows": expected_duplication_rows,
        "sample_duplication_count_pass": (
            len(duplication_df) == expected_duplication_rows
        ),

        "valid_duplication_source_indices": valid_source_index,
        "invalid_duplication_source_indices": invalid_source_index,
        "source_index_validation_pass": (
            valid_source_index == expected_duplication_rows
            and invalid_source_index == 0
        ),

        "sample_duplication_clean_feature_matches": feature_matches,
        "expected_feature_matches": expected_duplication_rows,
        "feature_match_pass": (
            feature_matches == expected_duplication_rows
        )
    })

duplicate_validation = pd.DataFrame(
    validation_rows
)

duplicate_validation["PHASE_2_6_PASS"] = (
    duplicate_validation[
        [
            "exact_duplicate_count_pass",
            "clean_duplicate_count_pass",
            "sample_duplication_count_pass",
            "source_index_validation_pass",
            "feature_match_pass"
        ]
    ]
    .all(axis=1)
)

duplicate_validation.to_csv(
    results_dir / "table_duplicate_validation.csv",
    index=False
)

print("\n" + "=" * 70)
print("PHASE 2.6 — FINAL DUPLICATE VALIDATION")
print("=" * 70)

print(
    duplicate_validation.to_string(
        index=False
    )
)

print("\nSaved:")
print(" - table_duplicate_validation.csv")


PHASE 2.6 — FINAL DUPLICATE VALIDATION
        dataset  exact_duplicate_rows  expected_exact_duplicate_rows  exact_duplicate_count_pass  clean_duplicate_rows  expected_clean_duplicate_rows  clean_duplicate_count_pass  sample_duplication_rows  expected_sample_duplication_rows  sample_duplication_count_pass  valid_duplication_source_indices  invalid_duplication_source_indices  source_index_validation_pass  sample_duplication_clean_feature_matches  expected_feature_matches  feature_match_pass  PHASE_2_6_PASS
CIC-MalMem-2022                   170                            170                        True                   163                            163                        True                     2342                              2342                           True                              2342                                   0                          True                                      2342                      2342                True            True
        Android 

Constant-Feature Audit

In [68]:
# Phase 2.7.1 — Define substantive numeric feature columns

metadata_cols = [
    "Class",
    "attack_type",
    "original_label",
    "source_index",
    "poisoned"
]

def get_numeric_features(df):
    return [
        c for c in df.columns
        if c not in metadata_cols
        and pd.api.types.is_numeric_dtype(df[c])
    ]

malmem_numeric_features = get_numeric_features(malmem)
android_numeric_features = get_numeric_features(android)

print("CIC-MalMem-2022 numeric feature count:",
      len(malmem_numeric_features))

print("Android numeric feature count:",
      len(android_numeric_features))

CIC-MalMem-2022 numeric feature count: 55
Android numeric feature count: 470


In [69]:
# Phase 2.7.2 — Constant numeric feature audit

def find_constant_features(df, feature_columns):
    return [
        c for c in feature_columns
        if df[c].nunique(dropna=False) <= 1
    ]

malmem_constant_features = find_constant_features(
    malmem,
    malmem_numeric_features
)

android_constant_features = find_constant_features(
    android,
    android_numeric_features
)

print("CIC-MalMem-2022 constant features:")
print(malmem_constant_features)

print("\nAndroid constant features:")
print(android_constant_features)

CIC-MalMem-2022 constant features:
['pslist.nprocs64bit', 'handles.nport', 'svcscan.interactive_process_services']

Android constant features:
['getGroupIdLevel1', 'registerSuggestionSpansForNotification']


In [70]:
# Phase 2.7.3 — Inspect values of constant features

print("CIC-MalMem-2022 constant feature values")

for feature in malmem_constant_features:
    print(
        f"{feature}: "
        f"unique_values={malmem[feature].unique()}"
    )

print("\nAndroid constant feature values")

for feature in android_constant_features:
    print(
        f"{feature}: "
        f"unique_values={android[feature].unique()}"
    )

CIC-MalMem-2022 constant feature values
pslist.nprocs64bit: unique_values=[0]
handles.nport: unique_values=[0]
svcscan.interactive_process_services: unique_values=[0]

Android constant feature values
getGroupIdLevel1: unique_values=[0.]
registerSuggestionSpansForNotification: unique_values=[0.]


In [71]:
# Phase 2.7.4 — Verify zero variance

malmem_variance = malmem[
    malmem_constant_features
].var(numeric_only=True)

android_variance = android[
    android_constant_features
].var(numeric_only=True)

print("CIC-MalMem-2022 constant-feature variance:")
print(malmem_variance)

print("\nAndroid constant-feature variance:")
print(android_variance)

CIC-MalMem-2022 constant-feature variance:
pslist.nprocs64bit                      0.0
handles.nport                           0.0
svcscan.interactive_process_services    0.0
dtype: float64

Android constant-feature variance:
getGroupIdLevel1                          0.0
registerSuggestionSpansForNotification    0.0
dtype: float64


In [72]:
# Phase 2.7.5 — Verify complete constant-feature set

malmem_nonconstant_features = [
    c for c in malmem_numeric_features
    if c not in malmem_constant_features
]

android_nonconstant_features = [
    c for c in android_numeric_features
    if c not in android_constant_features
]

malmem_remaining_constants = [
    c for c in malmem_nonconstant_features
    if malmem[c].nunique(dropna=False) <= 1
]

android_remaining_constants = [
    c for c in android_nonconstant_features
    if android[c].nunique(dropna=False) <= 1
]

print("MalMem unexpected remaining constants:")
print(malmem_remaining_constants)

print("\nAndroid unexpected remaining constants:")
print(android_remaining_constants)

MalMem unexpected remaining constants:
[]

Android unexpected remaining constants:
[]


In [73]:
# Phase 2.7.6 — Create modeling feature lists
# Constant features are excluded from modeling only.

malmem_model_features = [
    c for c in malmem_numeric_features
    if c not in malmem_constant_features
]

android_model_features = [
    c for c in android_numeric_features
    if c not in android_constant_features
]

print("CIC-MalMem-2022")
print("Original numeric features:", len(malmem_numeric_features))
print("Constant features excluded:", len(malmem_constant_features))
print("Final candidate modeling features:",
      len(malmem_model_features))

print("\nAndroid")
print("Original numeric features:", len(android_numeric_features))
print("Constant features excluded:", len(android_constant_features))
print("Final candidate modeling features:",
      len(android_model_features))

CIC-MalMem-2022
Original numeric features: 55
Constant features excluded: 3
Final candidate modeling features: 52

Android
Original numeric features: 470
Constant features excluded: 2
Final candidate modeling features: 468


In [74]:
# Phase 2.7.7 — Save constant-feature audit artifacts

results_dir = Path("../results/tables")
results_dir.mkdir(parents=True, exist_ok=True)

constant_rows = []

for feature in malmem_constant_features:
    constant_rows.append({
        "dataset": "CIC-MalMem-2022",
        "feature": feature,
        "unique_count": int(malmem[feature].nunique(dropna=False)),
        "variance": float(malmem[feature].var()),
        "constant": True,
        "modeling_decision": "exclude_from_modeling"
    })

for feature in android_constant_features:
    constant_rows.append({
        "dataset": "Android",
        "feature": feature,
        "unique_count": int(android[feature].nunique(dropna=False)),
        "variance": float(android[feature].var()),
        "constant": True,
        "modeling_decision": "exclude_from_modeling"
    })

constant_feature_audit = pd.DataFrame(constant_rows)

constant_feature_audit.to_csv(
    results_dir / "table_constant_feature_audit.csv",
    index=False
)

# Feature-count summary
feature_summary = pd.DataFrame([
    {
        "dataset": "CIC-MalMem-2022",
        "numeric_features_before": len(malmem_numeric_features),
        "constant_features": len(malmem_constant_features),
        "candidate_modeling_features": len(malmem_model_features)
    },
    {
        "dataset": "Android",
        "numeric_features_before": len(android_numeric_features),
        "constant_features": len(android_constant_features),
        "candidate_modeling_features": len(android_model_features)
    }
])

feature_summary.to_csv(
    results_dir / "table_feature_count_after_constant_audit.csv",
    index=False
)

print("Saved Phase 2.7 audit tables:")
print(" - table_constant_feature_audit.csv")
print(" - table_feature_count_after_constant_audit.csv")

Saved Phase 2.7 audit tables:
 - table_constant_feature_audit.csv
 - table_feature_count_after_constant_audit.csv


In [75]:
# Phase 2.7.8 — Final constant-feature validation

expected_constants = {
    "CIC-MalMem-2022": {
        "pslist.nprocs64bit",
        "handles.nport",
        "svcscan.interactive_process_services"
    },
    "Android": {
        "getGroupIdLevel1",
        "registerSuggestionSpansForNotification"
    }
}

actual_constants = {
    "CIC-MalMem-2022": set(malmem_constant_features),
    "Android": set(android_constant_features)
}

validation_rows = []

for dataset_name in [
    "CIC-MalMem-2022",
    "Android"
]:

    expected_set = expected_constants[dataset_name]
    actual_set = actual_constants[dataset_name]

    if dataset_name == "CIC-MalMem-2022":
        df = malmem
        numeric_features = malmem_numeric_features
        model_features = malmem_model_features
    else:
        df = android
        numeric_features = android_numeric_features
        model_features = android_model_features

    # Every detected constant must have exactly one value
    one_value_pass = all(
        df[c].nunique(dropna=False) == 1
        for c in actual_set
    )

    # Every detected constant must have zero variance
    zero_variance_pass = all(
        df[c].var() == 0
        for c in actual_set
    )

    # No unexpected constants
    complete_set_pass = actual_set == expected_set

    # Model features must contain no constants
    no_constants_in_model_features = all(
        df[c].nunique(dropna=False) > 1
        for c in model_features
    )

    validation_rows.append({
        "dataset": dataset_name,
        "detected_constant_features": len(actual_set),
        "expected_constant_features": len(expected_set),
        "constant_feature_count_pass": (
            len(actual_set) == len(expected_set)
        ),
        "constant_feature_set_pass": complete_set_pass,
        "one_unique_value_pass": one_value_pass,
        "zero_variance_pass": zero_variance_pass,
        "candidate_model_features": len(model_features),
        "no_constants_in_model_features": (
            no_constants_in_model_features
        )
    })

constant_validation = pd.DataFrame(validation_rows)

constant_validation["PHASE_2_7_PASS"] = (
    constant_validation[
        [
            "constant_feature_count_pass",
            "constant_feature_set_pass",
            "one_unique_value_pass",
            "zero_variance_pass",
            "no_constants_in_model_features"
        ]
    ].all(axis=1)
)

constant_validation.to_csv(
    results_dir / "table_constant_feature_validation.csv",
    index=False
)

print("\n" + "=" * 70)
print("PHASE 2.7 — FINAL CONSTANT-FEATURE VALIDATION")
print("=" * 70)

print(
    constant_validation.to_string(index=False)
)

print("\nSaved:")
print(" - table_constant_feature_validation.csv")


PHASE 2.7 — FINAL CONSTANT-FEATURE VALIDATION
        dataset  detected_constant_features  expected_constant_features  constant_feature_count_pass  constant_feature_set_pass  one_unique_value_pass  zero_variance_pass  candidate_model_features  no_constants_in_model_features  PHASE_2_7_PASS
CIC-MalMem-2022                           3                           3                         True                       True                   True                True                        52                            True            True
        Android                           2                           2                         True                       True                   True                True                       468                            True            True

Saved:
 - table_constant_feature_validation.csv


Impossible / Invalid Value Audit

In [76]:
# Phase 2.8.1 — Define substantive numeric features

metadata_cols = [
    "Class",
    "attack_type",
    "original_label",
    "source_index",
    "poisoned"
]

def get_numeric_features(df):
    return [
        c for c in df.columns
        if c not in metadata_cols
        and pd.api.types.is_numeric_dtype(df[c])
    ]

malmem_numeric_features = get_numeric_features(malmem)
android_numeric_features = get_numeric_features(android)

print("CIC-MalMem-2022 numeric features:",
      len(malmem_numeric_features))

print("Android numeric features:",
      len(android_numeric_features))

CIC-MalMem-2022 numeric features: 55
Android numeric features: 470


In [77]:
# Phase 2.8.2 — Negative-value audit

malmem_negative_by_feature = (
    (malmem[malmem_numeric_features] < 0)
    .sum()
    .sort_values(ascending=False)
)

android_negative_by_feature = (
    (android[android_numeric_features] < 0)
    .sum()
    .sort_values(ascending=False)
)

print("CIC-MalMem-2022 — negative values")
print(
    malmem_negative_by_feature[
        malmem_negative_by_feature > 0
    ]
)

print("\nAndroid — negative values")
print(
    android_negative_by_feature[
        android_negative_by_feature > 0
    ]
)

print("\nTotal negative cells — MalMem:",
      int(malmem_negative_by_feature.sum()))

print("Total negative cells — Android:",
      int(android_negative_by_feature.sum()))

CIC-MalMem-2022 — negative values
Series([], dtype: int64)

Android — negative values
Series([], dtype: int64)

Total negative cells — MalMem: 0
Total negative cells — Android: 0


In [78]:
# Phase 2.8.3 — Negative values by attack type

def negative_cells_by_attack(df, features):
    return (
        df.groupby("attack_type")
        .apply(
            lambda group: int(
                (group[features] < 0).sum().sum()
            ),
            include_groups=False
        )
        .sort_values(ascending=False)
    )

malmem_negative_by_attack = negative_cells_by_attack(
    malmem,
    malmem_numeric_features
)

android_negative_by_attack = negative_cells_by_attack(
    android,
    android_numeric_features
)

print("CIC-MalMem-2022 — negative cells by attack_type")
print(malmem_negative_by_attack)

print("\nAndroid — negative cells by attack_type")
print(android_negative_by_attack)

CIC-MalMem-2022 — negative cells by attack_type
attack_type
backdoor_injection          0
clean                       0
feature_poisoning           0
gaussian_noise_injection    0
label_flipping              0
missing_value_injection     0
outlier_injection           0
sample_duplication          0
dtype: int64

Android — negative cells by attack_type
attack_type
backdoor_injection          0
clean                       0
feature_poisoning           0
gaussian_noise_injection    0
label_flipping              0
missing_value_injection     0
outlier_injection           0
sample_duplication          0
dtype: int64


In [79]:
# Phase 2.8.4 — Infinite-value audit

malmem_inf_by_feature = np.isinf(
    malmem[malmem_numeric_features]
).sum().sort_values(ascending=False)

android_inf_by_feature = np.isinf(
    android[android_numeric_features]
).sum().sort_values(ascending=False)

print("CIC-MalMem-2022 — infinite values")
print(
    malmem_inf_by_feature[
        malmem_inf_by_feature > 0
    ]
)

print("\nAndroid — infinite values")
print(
    android_inf_by_feature[
        android_inf_by_feature > 0
    ]
)

print("\nTotal infinite cells — MalMem:",
      int(malmem_inf_by_feature.sum()))

print("Total infinite cells — Android:",
      int(android_inf_by_feature.sum()))

CIC-MalMem-2022 — infinite values
Series([], dtype: int64)

Android — infinite values
Series([], dtype: int64)

Total infinite cells — MalMem: 0
Total infinite cells — Android: 0


In [80]:
# Phase 2.8.5 — Non-finite value validation

def nonfinite_cells(df, features):
    numeric_data = df[features]

    return int(
        (~np.isfinite(numeric_data.fillna(0))).sum().sum()
    )

malmem_nonfinite = nonfinite_cells(
    malmem,
    malmem_numeric_features
)

android_nonfinite = nonfinite_cells(
    android,
    android_numeric_features
)

print("CIC-MalMem-2022 non-finite cells excluding NaN:",
      malmem_nonfinite)

print("Android non-finite cells excluding NaN:",
      android_nonfinite)

CIC-MalMem-2022 non-finite cells excluding NaN: 0
Android non-finite cells excluding NaN: 0


In [81]:
# Phase 2.8.6 — Clean-data invalid-value check

for dataset_name, df, features in [
    ("CIC-MalMem-2022", malmem, malmem_numeric_features),
    ("Android", android, android_numeric_features)
]:

    clean = df[
        df["attack_type"] == "clean"
    ]

    negative_cells = int(
        (clean[features] < 0).sum().sum()
    )

    infinite_cells = int(
        np.isinf(clean[features]).sum().sum()
    )

    print(f"\n{dataset_name}")
    print("Clean negative cells:", negative_cells)
    print("Clean infinite cells:", infinite_cells)


CIC-MalMem-2022
Clean negative cells: 0
Clean infinite cells: 0

Android
Clean negative cells: 0
Clean infinite cells: 0


In [82]:
# Phase 2.8.7 — Poisoned-data invalid-value check

for dataset_name, df, features in [
    ("CIC-MalMem-2022", malmem, malmem_numeric_features),
    ("Android", android, android_numeric_features)
]:

    poisoned = df[
        df["poisoned"] == 1
    ]

    negative_cells = int(
        (poisoned[features] < 0).sum().sum()
    )

    infinite_cells = int(
        np.isinf(poisoned[features]).sum().sum()
    )

    print(f"\n{dataset_name}")
    print("Poisoned negative cells:", negative_cells)
    print("Poisoned infinite cells:", infinite_cells)


CIC-MalMem-2022
Poisoned negative cells: 0
Poisoned infinite cells: 0

Android
Poisoned negative cells: 0
Poisoned infinite cells: 0


In [83]:
# Phase 2.8.8 — Save impossible/invalid value audit

results_dir = Path("../results/tables")
results_dir.mkdir(parents=True, exist_ok=True)

# Negative values by feature
negative_by_feature = pd.concat([
    malmem_negative_by_feature.rename(
        "negative_count"
    ).rename_axis("feature")
    .reset_index()
    .assign(dataset="CIC-MalMem-2022"),

    android_negative_by_feature.rename(
        "negative_count"
    ).rename_axis("feature")
    .reset_index()
    .assign(dataset="Android")
], ignore_index=True)

negative_by_feature = negative_by_feature[
    ["dataset", "feature", "negative_count"]
]

# Infinite values by feature
infinite_by_feature = pd.concat([
    malmem_inf_by_feature.rename(
        "infinite_count"
    ).rename_axis("feature")
    .reset_index()
    .assign(dataset="CIC-MalMem-2022"),

    android_inf_by_feature.rename(
        "infinite_count"
    ).rename_axis("feature")
    .reset_index()
    .assign(dataset="Android")
], ignore_index=True)

infinite_by_feature = infinite_by_feature[
    ["dataset", "feature", "infinite_count"]
]

# Negative values by attack
negative_by_attack = pd.concat([
    malmem_negative_by_attack.rename(
        "negative_cells"
    ).rename_axis("attack_type")
    .reset_index()
    .assign(dataset="CIC-MalMem-2022"),

    android_negative_by_attack.rename(
        "negative_cells"
    ).rename_axis("attack_type")
    .reset_index()
    .assign(dataset="Android")
], ignore_index=True)

negative_by_attack = negative_by_attack[
    ["dataset", "attack_type", "negative_cells"]
]

negative_by_feature.to_csv(
    results_dir / "table_negative_values_by_feature.csv",
    index=False
)

infinite_by_feature.to_csv(
    results_dir / "table_infinite_values_by_feature.csv",
    index=False
)

negative_by_attack.to_csv(
    results_dir / "table_negative_values_by_attack.csv",
    index=False
)

print("Saved Phase 2.8 audit tables:")
print(" - table_negative_values_by_feature.csv")
print(" - table_infinite_values_by_feature.csv")
print(" - table_negative_values_by_attack.csv")

Saved Phase 2.8 audit tables:
 - table_negative_values_by_feature.csv
 - table_infinite_values_by_feature.csv
 - table_negative_values_by_attack.csv


In [84]:
# Phase 2.8.9 — Final impossible/invalid-value validation

validation_rows = []

for dataset_name, df, features in [
    ("CIC-MalMem-2022", malmem, malmem_numeric_features),
    ("Android", android, android_numeric_features)
]:

    negative_cells = int(
        (df[features] < 0).sum().sum()
    )

    infinite_cells = int(
        np.isinf(df[features]).sum().sum()
    )

    clean = df[
        df["attack_type"] == "clean"
    ]

    clean_negative_cells = int(
        (clean[features] < 0).sum().sum()
    )

    clean_infinite_cells = int(
        np.isinf(clean[features]).sum().sum()
    )

    poisoned = df[
        df["poisoned"] == 1
    ]

    poisoned_negative_cells = int(
        (poisoned[features] < 0).sum().sum()
    )

    poisoned_infinite_cells = int(
        np.isinf(poisoned[features]).sum().sum()
    )

    validation_rows.append({
        "dataset": dataset_name,

        "numeric_feature_count": len(features),

        "negative_cells": negative_cells,
        "negative_value_pass": (
            negative_cells == 0
        ),

        "infinite_cells": infinite_cells,
        "infinite_value_pass": (
            infinite_cells == 0
        ),

        "clean_negative_cells": clean_negative_cells,
        "clean_negative_pass": (
            clean_negative_cells == 0
        ),

        "clean_infinite_cells": clean_infinite_cells,
        "clean_infinite_pass": (
            clean_infinite_cells == 0
        ),

        "poisoned_negative_cells": poisoned_negative_cells,
        "poisoned_negative_pass": (
            poisoned_negative_cells == 0
        ),

        "poisoned_infinite_cells": poisoned_infinite_cells,
        "poisoned_infinite_pass": (
            poisoned_infinite_cells == 0
        )
    })

invalid_validation = pd.DataFrame(
    validation_rows
)

invalid_validation["PHASE_2_8_PASS"] = (
    invalid_validation[
        [
            "negative_value_pass",
            "infinite_value_pass",
            "clean_negative_pass",
            "clean_infinite_pass",
            "poisoned_negative_pass",
            "poisoned_infinite_pass"
        ]
    ].all(axis=1)
)

invalid_validation.to_csv(
    results_dir / "table_invalid_value_validation.csv",
    index=False
)

print("\n" + "=" * 70)
print("PHASE 2.8 — FINAL IMPOSSIBLE/INVALID VALUE VALIDATION")
print("=" * 70)

print(
    invalid_validation.to_string(
        index=False
    )
)

print("\nSaved:")
print(" - table_invalid_value_validation.csv")


PHASE 2.8 — FINAL IMPOSSIBLE/INVALID VALUE VALIDATION
        dataset  numeric_feature_count  negative_cells  negative_value_pass  infinite_cells  infinite_value_pass  clean_negative_cells  clean_negative_pass  clean_infinite_cells  clean_infinite_pass  poisoned_negative_cells  poisoned_negative_pass  poisoned_infinite_cells  poisoned_infinite_pass  PHASE_2_8_PASS
CIC-MalMem-2022                     55               0                 True               0                 True                     0                 True                     0                 True                        0                    True                        0                    True            True
        Android                    470               0                 True               0                 True                     0                 True                     0                 True                        0                    True                        0                    True            True

Saved

Outlier Audit

In [85]:
# Phase 2.9.1 — Define candidate modeling features

metadata_cols = [
    "Class",
    "attack_type",
    "original_label",
    "source_index",
    "poisoned"
]

def get_numeric_features(df):
    return [
        c for c in df.columns
        if c not in metadata_cols
        and pd.api.types.is_numeric_dtype(df[c])
    ]

def get_nonconstant_features(df, numeric_features):
    return [
        c for c in numeric_features
        if df[c].nunique(dropna=False) > 1
    ]

malmem_numeric_features = get_numeric_features(malmem)
android_numeric_features = get_numeric_features(android)

malmem_model_features = get_nonconstant_features(
    malmem,
    malmem_numeric_features
)

android_model_features = get_nonconstant_features(
    android,
    android_numeric_features
)

print("CIC-MalMem-2022 candidate features:",
      len(malmem_model_features))

print("Android candidate features:",
      len(android_model_features))

CIC-MalMem-2022 candidate features: 52
Android candidate features: 468


In [86]:
# Phase 2.9.2 — IQR-based outlier counts

def calculate_iqr_outliers(df, features):

    numeric = df[features]

    q1 = numeric.quantile(0.25)
    q3 = numeric.quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (
        (numeric < lower_bound)
        | (numeric > upper_bound)
    )

    outlier_count = (
        outlier_mask
        .sum()
        .sort_values(ascending=False)
    )

    return (
        outlier_count,
        lower_bound,
        upper_bound
    )


(
    malmem_iqr_outliers,
    malmem_lower_bounds,
    malmem_upper_bounds
) = calculate_iqr_outliers(
    malmem,
    malmem_model_features
)

(
    android_iqr_outliers,
    android_lower_bounds,
    android_upper_bounds
) = calculate_iqr_outliers(
    android,
    android_model_features
)

print("CIC-MalMem-2022 — Top 20 features by IQR outlier count")
print(malmem_iqr_outliers.head(20))

print("\nAndroid — Top 20 features by IQR outlier count")
print(android_iqr_outliers.head(20))

CIC-MalMem-2022 — Top 20 features by IQR outlier count
malfind.commitCharge                      8884
handles.ndesktop                          6898
svcscan.nactive                           6665
ldrmodules.not_in_init_avg                6618
pslist.nproc                              6251
malfind.protection                        5822
malfind.ninjections                       5712
pslist.avg_handlers                       4736
psxview.not_in_deskthrd                   4477
modules.nmodules                          4449
handles.ndirectory                        4449
psxview.not_in_ethread_pool               4339
psxview.not_in_csrss_handles              4337
handles.nhandles                          3859
malfind.uniqueInjections                  3847
psxview.not_in_ethread_pool_false_avg     3509
psxview.not_in_pspcid_list                3294
psxview.not_in_pslist                     3287
psxview.not_in_session                    3286
psxview.not_in_csrss_handles_false_avg    3212
dtype

In [87]:
# Phase 2.9.3 — Outlier percentages

malmem_outlier_percentage = (
    malmem_iqr_outliers / len(malmem) * 100
).sort_values(ascending=False)

android_outlier_percentage = (
    android_iqr_outliers / len(android) * 100
).sort_values(ascending=False)

print("CIC-MalMem-2022 — Top 20 outlier percentages")
print(
    malmem_outlier_percentage.head(20)
)

print("\nAndroid — Top 20 outlier percentages")
print(
    android_outlier_percentage.head(20)
)

CIC-MalMem-2022 — Top 20 outlier percentages
malfind.commitCharge                      15.161444
handles.ndesktop                          11.772135
svcscan.nactive                           11.374497
ldrmodules.not_in_init_avg                11.294286
pslist.nproc                              10.667964
malfind.protection                         9.935832
malfind.ninjections                        9.748106
pslist.avg_handlers                        8.082463
psxview.not_in_deskthrd                    7.640453
modules.nmodules                           7.592668
handles.ndirectory                         7.592668
psxview.not_in_ethread_pool                7.404942
psxview.not_in_csrss_handles               7.401529
handles.nhandles                           6.585774
malfind.uniqueInjections                   6.565295
psxview.not_in_ethread_pool_false_avg      5.988463
psxview.not_in_pspcid_list                 5.621544
psxview.not_in_pslist                      5.609598
psxview.not_in_sess

In [88]:
# Phase 2.9.4 — IQR outlier counts by attack type

def outlier_counts_by_attack(
    df,
    features,
    lower_bounds,
    upper_bounds
):

    results = []

    for attack_type, group in df.groupby("attack_type"):

        values = group[features]

        outlier_mask = (
            (values < lower_bounds)
            | (values > upper_bounds)
        )

        total_outliers = int(
            outlier_mask.sum().sum()
        )

        affected_rows = int(
            outlier_mask.any(axis=1).sum()
        )

        results.append({
            "attack_type": attack_type,
            "outlier_cells": total_outliers,
            "rows_with_outlier": affected_rows,
            "observations": len(group),
            "row_outlier_percentage": (
                affected_rows / len(group) * 100
            )
        })

    return pd.DataFrame(results).sort_values(
        "outlier_cells",
        ascending=False
    )


malmem_outliers_by_attack = outlier_counts_by_attack(
    malmem,
    malmem_model_features,
    malmem_lower_bounds,
    malmem_upper_bounds
)

android_outliers_by_attack = outlier_counts_by_attack(
    android,
    android_model_features,
    android_lower_bounds,
    android_upper_bounds
)

print("CIC-MalMem-2022 — outliers by attack_type")
print(malmem_outliers_by_attack)

print("\nAndroid — outliers by attack_type")
print(android_outliers_by_attack)

CIC-MalMem-2022 — outliers by attack_type
                attack_type  outlier_cells  rows_with_outlier  observations  \
1                     clean          78644              12801         42202   
3  gaussian_noise_injection          13069               2342          2342   
6         outlier_injection          10319               2342          2342   
2         feature_poisoning           9597               2342          2342   
0        backdoor_injection           7961               2342          2342   
5   missing_value_injection           3364                669          2342   
7        sample_duplication           3339                701          2342   
4            label_flipping           3086                670          2342   

   row_outlier_percentage  
1               30.332686  
3              100.000000  
6              100.000000  
2              100.000000  
0              100.000000  
5               28.565329  
7               29.931682  
4               28.608

In [89]:
# Phase 2.9.5 — Clean vs poisoned outlier prevalence

def clean_vs_poisoned_outliers(
    df,
    features,
    lower_bounds,
    upper_bounds
):

    values = df[features]

    outlier_mask = (
        (values < lower_bounds)
        | (values > upper_bounds)
    )

    row_has_outlier = outlier_mask.any(axis=1)

    result = []

    for group_name, mask in [
        ("clean", df["poisoned"] == 0),
        ("poisoned", df["poisoned"] == 1)
    ]:

        observations = int(mask.sum())

        affected_rows = int(
            row_has_outlier[mask].sum()
        )

        result.append({
            "group": group_name,
            "observations": observations,
            "rows_with_outlier": affected_rows,
            "row_outlier_percentage": (
                affected_rows / observations * 100
                if observations > 0 else 0
            )
        })

    return pd.DataFrame(result)


malmem_clean_poisoned_outliers = clean_vs_poisoned_outliers(
    malmem,
    malmem_model_features,
    malmem_lower_bounds,
    malmem_upper_bounds
)

android_clean_poisoned_outliers = clean_vs_poisoned_outliers(
    android,
    android_model_features,
    android_lower_bounds,
    android_upper_bounds
)

print("CIC-MalMem-2022")
print(malmem_clean_poisoned_outliers)

print("\nAndroid")
print(android_clean_poisoned_outliers)

CIC-MalMem-2022
      group  observations  rows_with_outlier  row_outlier_percentage
0     clean         42202              12801               30.332686
1  poisoned         16394              11408               69.586434

Android
      group  observations  rows_with_outlier  row_outlier_percentage
0     clean          6117               5340               87.297695
1  poisoned          5481               4818               87.903667


Examine outlier_injection specifically

In [90]:
# Phase 2.9.6 — Specific audit of outlier_injection

for dataset_name, df, features, lower, upper in [
    (
        "CIC-MalMem-2022",
        malmem,
        malmem_model_features,
        malmem_lower_bounds,
        malmem_upper_bounds
    ),
    (
        "Android",
        android,
        android_model_features,
        android_lower_bounds,
        android_upper_bounds
    )
]:

    attack_df = df[
        df["attack_type"] == "outlier_injection"
    ]

    values = attack_df[features]

    outlier_mask = (
        (values < lower)
        | (values > upper)
    )

    affected_rows = int(
        outlier_mask.any(axis=1).sum()
    )

    outlier_cells = int(
        outlier_mask.sum().sum()
    )

    print(f"\n{dataset_name}")
    print("outlier_injection observations:",
          len(attack_df))
    print("Rows containing IQR outliers:",
          affected_rows)
    print("IQR outlier cells:",
          outlier_cells)


CIC-MalMem-2022
outlier_injection observations: 2342
Rows containing IQR outliers: 2342
IQR outlier cells: 10319

Android
outlier_injection observations: 783
Rows containing IQR outliers: 783
IQR outlier cells: 14105


In [91]:
# Phase 2.9.7 — Save outlier audit artifacts

results_dir = Path("../results/tables")
results_dir.mkdir(parents=True, exist_ok=True)

# Feature-level counts
outlier_by_feature = pd.concat([
    pd.DataFrame({
        "dataset": "CIC-MalMem-2022",
        "feature": malmem_iqr_outliers.index,
        "outlier_count": malmem_iqr_outliers.values,
        "outlier_percentage": malmem_outlier_percentage[
            malmem_iqr_outliers.index
        ].values
    }),

    pd.DataFrame({
        "dataset": "Android",
        "feature": android_iqr_outliers.index,
        "outlier_count": android_iqr_outliers.values,
        "outlier_percentage": android_outlier_percentage[
            android_iqr_outliers.index
        ].values
    })
], ignore_index=True)

# Attack-level counts
outlier_by_attack = pd.concat([
    malmem_outliers_by_attack.assign(
        dataset="CIC-MalMem-2022"
    ),
    android_outliers_by_attack.assign(
        dataset="Android"
    )
], ignore_index=True)

outlier_by_attack = outlier_by_attack[
    [
        "dataset",
        "attack_type",
        "observations",
        "outlier_cells",
        "rows_with_outlier",
        "row_outlier_percentage"
    ]
]

# Clean vs poisoned
outlier_clean_poisoned = pd.concat([
    malmem_clean_poisoned_outliers.assign(
        dataset="CIC-MalMem-2022"
    ),
    android_clean_poisoned_outliers.assign(
        dataset="Android"
    )
], ignore_index=True)

outlier_by_feature.to_csv(
    results_dir / "table_outlier_by_feature.csv",
    index=False
)

outlier_by_attack.to_csv(
    results_dir / "table_outlier_by_attack.csv",
    index=False
)

outlier_clean_poisoned.to_csv(
    results_dir / "table_outlier_clean_vs_poisoned.csv",
    index=False
)

print("Saved Phase 2.9 audit tables:")
print(" - table_outlier_by_feature.csv")
print(" - table_outlier_by_attack.csv")
print(" - table_outlier_clean_vs_poisoned.csv")

Saved Phase 2.9 audit tables:
 - table_outlier_by_feature.csv
 - table_outlier_by_attack.csv
 - table_outlier_clean_vs_poisoned.csv


In [92]:
# Phase 2.9.8 — Final outlier audit validation

validation_rows = []

for dataset_name, df, features, outliers in [
    (
        "CIC-MalMem-2022",
        malmem,
        malmem_model_features,
        malmem_iqr_outliers
    ),
    (
        "Android",
        android,
        android_model_features,
        android_iqr_outliers
    )
]:

    # Number of candidate features
    feature_count_pass = len(features) > 0

    # Outlier counts must be non-negative
    nonnegative_counts_pass = bool(
        (outliers >= 0).all()
    )

    # Every feature in the audit must belong to candidate features
    feature_alignment_pass = (
        set(outliers.index) == set(features)
    )

    # Verify attack-level audit covers every attack type
    attack_audit = (
        malmem_outliers_by_attack
        if dataset_name == "CIC-MalMem-2022"
        else android_outliers_by_attack
    )

    attack_coverage_pass = (
        set(attack_audit["attack_type"])
        == set(df["attack_type"].unique())
    )

    validation_rows.append({
        "dataset": dataset_name,
        "candidate_features": len(features),
        "feature_count_pass": feature_count_pass,
        "nonnegative_outlier_counts_pass": nonnegative_counts_pass,
        "feature_alignment_pass": feature_alignment_pass,
        "attack_type_coverage_pass": attack_coverage_pass,

        # Explicit methodological rule:
        # outliers are NOT automatically removed
        "automatic_removal": False
    })

outlier_validation = pd.DataFrame(
    validation_rows
)

outlier_validation["PHASE_2_9_PASS"] = (
    outlier_validation[
        [
            "feature_count_pass",
            "nonnegative_outlier_counts_pass",
            "feature_alignment_pass",
            "attack_type_coverage_pass",
            "automatic_removal"
        ]
    ].copy()
    .assign(automatic_removal=lambda x: ~x["automatic_removal"])
    .all(axis=1)
)

outlier_validation.to_csv(
    results_dir / "table_outlier_validation.csv",
    index=False
)

print("\n" + "=" * 70)
print("PHASE 2.9 — FINAL OUTLIER AUDIT VALIDATION")
print("=" * 70)

print(
    outlier_validation.to_string(
        index=False
    )
)

print("\nSaved:")
print(" - table_outlier_validation.csv")


PHASE 2.9 — FINAL OUTLIER AUDIT VALIDATION
        dataset  candidate_features  feature_count_pass  nonnegative_outlier_counts_pass  feature_alignment_pass  attack_type_coverage_pass  automatic_removal  PHASE_2_9_PASS
CIC-MalMem-2022                  52                True                             True                    True                       True              False            True
        Android                 468                True                             True                    True                       True              False            True

Saved:
 - table_outlier_validation.csv


Separate Class-Imbalance Audit

In [93]:
# Phase 2.10.1 — Load separate class-imbalance audit files

malmem_imbalance_path = (
    "../data/raw/cic_malmem2022_removed_audit.csv"
)

android_imbalance_path = (
    "../data/raw/cccs_andmal2020_removed_audit.csv"
)

malmem_imbalance = pd.read_csv(
    malmem_imbalance_path
)

android_imbalance = pd.read_csv(
    android_imbalance_path
)

print("CIC-MalMem-2022 class-imbalance audit:")
print(malmem_imbalance.shape)

print("\nAndroid class-imbalance audit:")
print(android_imbalance.shape)

CIC-MalMem-2022 class-imbalance audit:
(2342, 59)

Android class-imbalance audit:
(783, 474)


In [94]:
# Phase 2.10.2 — Verify metadata structure

print("CIC-MalMem-2022 columns:")
print(malmem_imbalance.columns.tolist())

print("\nAndroid columns:")
print(android_imbalance.columns.tolist())

print("\nMalMem has attack_type:",
      "attack_type" in malmem_imbalance.columns)

print("MalMem has source_index:",
      "source_index" in malmem_imbalance.columns)

print("\nAndroid has attack_type:",
      "attack_type" in android_imbalance.columns)

print("Android has source_index:",
      "source_index" in android_imbalance.columns)

print("\nAndroid label column:")
print(
    [
        c for c in ["Class", "Cat"]
        if c in android_imbalance.columns
    ]
)

CIC-MalMem-2022 columns:
['Cat', 'pslist.nproc', 'pslist.nppid', 'pslist.avg_threads', 'pslist.nprocs64bit', 'pslist.avg_handlers', 'dlllist.ndlls', 'dlllist.avg_dlls_per_proc', 'handles.nhandles', 'handles.avg_handles_per_proc', 'handles.nport', 'handles.nfile', 'handles.nevent', 'handles.ndesktop', 'handles.nkey', 'handles.nthread', 'handles.ndirectory', 'handles.nsemaphore', 'handles.ntimer', 'handles.nsection', 'handles.nmutant', 'ldrmodules.not_in_load', 'ldrmodules.not_in_init', 'ldrmodules.not_in_mem', 'ldrmodules.not_in_load_avg', 'ldrmodules.not_in_init_avg', 'ldrmodules.not_in_mem_avg', 'malfind.ninjections', 'malfind.commitCharge', 'malfind.protection', 'malfind.uniqueInjections', 'psxview.not_in_pslist', 'psxview.not_in_eprocess_pool', 'psxview.not_in_ethread_pool', 'psxview.not_in_pspcid_list', 'psxview.not_in_csrss_handles', 'psxview.not_in_session', 'psxview.not_in_deskthrd', 'psxview.not_in_pslist_false_avg', 'psxview.not_in_eprocess_pool_false_avg', 'psxview.not_in_eth

In [100]:
# Phase 2.10.3 — Verify class-imbalance attack type

print("CIC-MalMem-2022 attack types:")
print(
    malmem_imbalance["attack_type"]
    .value_counts(dropna=False)
)

print("\nAndroid attack types:")
print(
    android_imbalance["attack_type"]
    .value_counts(dropna=False)
)

CIC-MalMem-2022 attack types:
attack_type
class_imbalance_attack    2342
Name: count, dtype: int64

Android attack types:
attack_type
class_imbalance_attack    783
Name: count, dtype: int64


In [101]:
# Phase 2.10.4 — Inspect and identify class column
# DO NOT assume Class or Cat

print("=" * 70)
print("CIC-MalMem-2022 class-imbalance columns")
print("=" * 70)

print(malmem_imbalance.columns.tolist())


print("\n" + "=" * 70)
print("Android class-imbalance columns")
print("=" * 70)

print(android_imbalance.columns.tolist())


# ---------------------------------------------------------
# Show columns that could plausibly contain class labels
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("Potential label columns")
print("=" * 70)

for dataset_name, df in [
    ("CIC-MalMem-2022", malmem_imbalance),
    ("Android", android_imbalance)
]:

    candidates = []

    for column in df.columns:
        name = str(column).strip().lower()

        if (
            "class" in name
            or "label" in name
            or name in ["cat", "category", "target", "y"]
        ):
            candidates.append(column)

    print(f"\n{dataset_name}:")
    print(candidates)

    for column in candidates:
        print(f"\n{column} value counts:")
        print(
            df[column]
            .value_counts(dropna=False)
            .head(20)
        )

CIC-MalMem-2022 class-imbalance columns
['Cat', 'pslist.nproc', 'pslist.nppid', 'pslist.avg_threads', 'pslist.nprocs64bit', 'pslist.avg_handlers', 'dlllist.ndlls', 'dlllist.avg_dlls_per_proc', 'handles.nhandles', 'handles.avg_handles_per_proc', 'handles.nport', 'handles.nfile', 'handles.nevent', 'handles.ndesktop', 'handles.nkey', 'handles.nthread', 'handles.ndirectory', 'handles.nsemaphore', 'handles.ntimer', 'handles.nsection', 'handles.nmutant', 'ldrmodules.not_in_load', 'ldrmodules.not_in_init', 'ldrmodules.not_in_mem', 'ldrmodules.not_in_load_avg', 'ldrmodules.not_in_init_avg', 'ldrmodules.not_in_mem_avg', 'malfind.ninjections', 'malfind.commitCharge', 'malfind.protection', 'malfind.uniqueInjections', 'psxview.not_in_pslist', 'psxview.not_in_eprocess_pool', 'psxview.not_in_ethread_pool', 'psxview.not_in_pspcid_list', 'psxview.not_in_csrss_handles', 'psxview.not_in_session', 'psxview.not_in_deskthrd', 'psxview.not_in_pslist_false_avg', 'psxview.not_in_eprocess_pool_false_avg', 'psx

In [102]:
# Phase 2.10.5 — Missing-value audit for class-imbalance files

for dataset_name, df in {
    "CIC-MalMem-2022": malmem_imbalance,
    "Android": android_imbalance
}.items():

    missing_cells = int(
        df.isna().sum().sum()
    )

    rows_with_missing = int(
        df.isna().any(axis=1).sum()
    )

    print(f"\n{dataset_name}")
    print("Missing cells:", missing_cells)
    print("Rows with missing values:", rows_with_missing)


CIC-MalMem-2022
Missing cells: 0
Rows with missing values: 0

Android
Missing cells: 0
Rows with missing values: 0


In [103]:
# Phase 2.10.6 — Source-index audit for class-imbalance files

for dataset_name, df in {
    "CIC-MalMem-2022": malmem_imbalance,
    "Android": android_imbalance
}.items():

    valid_source_index = int(
        (
            df["source_index"].notna()
            & (df["source_index"] >= 0)
        ).sum()
    )

    negative_source_index = int(
        (
            df["source_index"] < 0
        ).sum()
    )

    missing_source_index = int(
        df["source_index"].isna().sum()
    )

    print(f"\n{dataset_name}")
    print("Valid/non-negative source_index:",
          valid_source_index)
    print("Negative source_index:",
          negative_source_index)
    print("Missing source_index:",
          missing_source_index)


CIC-MalMem-2022
Valid/non-negative source_index: 0
Negative source_index: 2342
Missing source_index: 0

Android
Valid/non-negative source_index: 0
Negative source_index: 783
Missing source_index: 0


In [104]:
# Phase 2.10.7 — Exact duplicate audit

for dataset_name, df in {
    "CIC-MalMem-2022": malmem_imbalance,
    "Android": android_imbalance
}.items():

    exact_duplicates = int(
        df.duplicated().sum()
    )

    print(f"{dataset_name}")
    print(
        "Exact duplicate rows:",
        exact_duplicates
    )

CIC-MalMem-2022
Exact duplicate rows: 3
Android
Exact duplicate rows: 0


In [105]:
# Phase 2.10.8 — Compare class-imbalance audit
# with corresponding primary dataset

def get_feature_columns_for_comparison(df):
    excluded = [
        "Class",
        "Cat",
        "attack_type",
        "original_label",
        "source_index",
        "poisoned"
    ]

    return [
        c for c in df.columns
        if c not in excluded
        and pd.api.types.is_numeric_dtype(df[c])
    ]


# ---------------- MalMem ----------------

malmem_features_primary = get_feature_columns_for_comparison(
    malmem
)

malmem_features_imbalance = get_feature_columns_for_comparison(
    malmem_imbalance
)

common_malmem_features = [
    c for c in malmem_features_primary
    if c in malmem_features_imbalance
]

primary_malmem_hashes = set(
    pd.util.hash_pandas_object(
        malmem[common_malmem_features],
        index=False
    )
)

imbalance_malmem_hashes = pd.util.hash_pandas_object(
    malmem_imbalance[common_malmem_features],
    index=False
)

malmem_feature_overlap = int(
    imbalance_malmem_hashes.isin(
        primary_malmem_hashes
    ).sum()
)


# ---------------- Android ----------------

android_features_primary = get_feature_columns_for_comparison(
    android
)

android_features_imbalance = get_feature_columns_for_comparison(
    android_imbalance
)

common_android_features = [
    c for c in android_features_primary
    if c in android_features_imbalance
]

primary_android_hashes = set(
    pd.util.hash_pandas_object(
        android[common_android_features],
        index=False
    )
)

imbalance_android_hashes = pd.util.hash_pandas_object(
    android_imbalance[common_android_features],
    index=False
)

android_feature_overlap = int(
    imbalance_android_hashes.isin(
        primary_android_hashes
    ).sum()
)

print("CIC-MalMem-2022")
print("Common feature count:",
      len(common_malmem_features))
print(
    "Exact feature-vector overlap with primary dataset:",
    malmem_feature_overlap
)

print("\nAndroid")
print("Common feature count:",
      len(common_android_features))
print(
    "Exact feature-vector overlap with primary dataset:",
    android_feature_overlap
)

CIC-MalMem-2022
Common feature count: 55
Exact feature-vector overlap with primary dataset: 0

Android
Common feature count: 470
Exact feature-vector overlap with primary dataset: 5


In [106]:
# Phase 2.10.9 — Confirm separation from primary attacks

primary_attack_types = set(
    malmem["attack_type"].unique()
)

primary_attack_types_android = set(
    android["attack_type"].unique()
)

malmem_imbalance_attack_types = set(
    malmem_imbalance["attack_type"].unique()
)

android_imbalance_attack_types = set(
    android_imbalance["attack_type"].unique()
)

print("Primary MalMem attack types:")
print(sorted(primary_attack_types))

print("\nSeparate MalMem audit attack types:")
print(sorted(malmem_imbalance_attack_types))

print("\nPrimary Android attack types:")
print(sorted(primary_attack_types_android))

print("\nSeparate Android audit attack types:")
print(sorted(android_imbalance_attack_types))

print("\nClass-imbalance attack is separate from primary MalMem attacks:",
      malmem_imbalance_attack_types.isdisjoint(
          primary_attack_types
      ))

print(
    "Class-imbalance attack is separate from primary Android attacks:",
    android_imbalance_attack_types.isdisjoint(
        primary_attack_types_android
    )
)

Primary MalMem attack types:
['backdoor_injection', 'clean', 'feature_poisoning', 'gaussian_noise_injection', 'label_flipping', 'missing_value_injection', 'outlier_injection', 'sample_duplication']

Separate MalMem audit attack types:
['class_imbalance_attack']

Primary Android attack types:
['backdoor_injection', 'clean', 'feature_poisoning', 'gaussian_noise_injection', 'label_flipping', 'missing_value_injection', 'outlier_injection', 'sample_duplication']

Separate Android audit attack types:
['class_imbalance_attack']

Class-imbalance attack is separate from primary MalMem attacks: True
Class-imbalance attack is separate from primary Android attacks: True


In [108]:
# Phase 2.10.10 — Save class-imbalance audit artifacts
# Robust version: automatically detects the label column

results_dir = Path("../results/tables")
results_dir.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. Automatically identify the observed class/label column
# ============================================================

def detect_label_column(df, dataset_name):

    # Known metadata columns that are NOT class labels
    excluded = {
        "attack_type",
        "source_index",
        "original_label",
        "poisoned"
    }

    # Prefer non-numeric columns because class labels are categorical
    categorical_candidates = [
        col for col in df.columns
        if col not in excluded
        and not pd.api.types.is_numeric_dtype(df[col])
    ]

    # Common label names get priority
    preferred_names = [
        "Class",
        "class",
        "Cat",
        "cat",
        "Label",
        "label",
        "Category",
        "category",
        "target",
        "Target"
    ]

    for name in preferred_names:
        if name in categorical_candidates:
            return name

    # If exactly one categorical candidate remains,
    # use it automatically
    if len(categorical_candidates) == 1:
        return categorical_candidates[0]

    raise ValueError(
        f"Could not uniquely identify the class/label column "
        f"for {dataset_name}.\n"
        f"Categorical candidates found: {categorical_candidates}\n"
        f"All columns: {df.columns.tolist()}"
    )


malmem_label_column = detect_label_column(
    malmem_imbalance,
    "CIC-MalMem-2022"
)

android_label_column = detect_label_column(
    android_imbalance,
    "Android"
)


print("Detected label columns:")
print(
    "CIC-MalMem-2022:",
    malmem_label_column
)
print(
    "Android:",
    android_label_column
)


# ============================================================
# 2. Attack-type summary
# ============================================================

imbalance_attack_summary = pd.DataFrame([
    {
        "dataset": "CIC-MalMem-2022",
        "rows": len(malmem_imbalance),
        "attack_type": (
            malmem_imbalance["attack_type"]
            .iloc[0]
        )
    },
    {
        "dataset": "Android",
        "rows": len(android_imbalance),
        "attack_type": (
            android_imbalance["attack_type"]
            .iloc[0]
        )
    }
])


# ============================================================
# 3. Class distributions
# ============================================================

malmem_class_distribution = (
    malmem_imbalance[
        malmem_label_column
    ]
    .value_counts(dropna=False)
    .rename("observations")
    .rename_axis("class")
    .reset_index()
)

malmem_class_distribution["dataset"] = (
    "CIC-MalMem-2022"
)


android_class_distribution = (
    android_imbalance[
        android_label_column
    ]
    .value_counts(dropna=False)
    .rename("observations")
    .rename_axis("class")
    .reset_index()
)

android_class_distribution["dataset"] = (
    "Android"
)


imbalance_class_distribution = pd.concat(
    [
        malmem_class_distribution,
        android_class_distribution
    ],
    ignore_index=True
)

imbalance_class_distribution = (
    imbalance_class_distribution[
        ["dataset", "class", "observations"]
    ]
)


# ============================================================
# 4. Data-quality summary
# ============================================================

imbalance_quality = pd.DataFrame([
    {
        "dataset": "CIC-MalMem-2022",
        "rows": len(malmem_imbalance),

        "missing_cells": int(
            malmem_imbalance.isna().sum().sum()
        ),

        "exact_duplicates": int(
            malmem_imbalance.duplicated().sum()
        ),

        "nonnegative_source_indices": int(
            (
                malmem_imbalance["source_index"] >= 0
            ).sum()
        ),

        "feature_overlap_with_primary":
            malmem_feature_overlap
    },

    {
        "dataset": "Android",
        "rows": len(android_imbalance),

        "missing_cells": int(
            android_imbalance.isna().sum().sum()
        ),

        "exact_duplicates": int(
            android_imbalance.duplicated().sum()
        ),

        "nonnegative_source_indices": int(
            (
                android_imbalance["source_index"] >= 0
            ).sum()
        ),

        "feature_overlap_with_primary":
            android_feature_overlap
    }
])


# ============================================================
# 5. Save artifacts
# ============================================================

imbalance_attack_summary.to_csv(
    results_dir / "table_class_imbalance_attack_summary.csv",
    index=False
)

imbalance_class_distribution.to_csv(
    results_dir / "table_class_imbalance_class_distribution.csv",
    index=False
)

imbalance_quality.to_csv(
    results_dir / "table_class_imbalance_quality_audit.csv",
    index=False
)


print("\nSaved Phase 2.10 audit tables:")
print(" - table_class_imbalance_attack_summary.csv")
print(" - table_class_imbalance_class_distribution.csv")
print(" - table_class_imbalance_quality_audit.csv")

Detected label columns:
CIC-MalMem-2022: Cat
Android: Class

Saved Phase 2.10 audit tables:
 - table_class_imbalance_attack_summary.csv
 - table_class_imbalance_class_distribution.csv
 - table_class_imbalance_quality_audit.csv


In [109]:
# Phase 2.10.11 — Final class-imbalance audit validation

validation_rows = []

# ---------------- MalMem ----------------

malmem_attack_only = (
    malmem_imbalance["attack_type"].nunique() == 1
    and
    malmem_imbalance["attack_type"].iloc[0]
    == "class_imbalance_attack"
)

malmem_rows_pass = (
    len(malmem_imbalance) == 2342
)

malmem_missing_pass = (
    malmem_imbalance.isna().sum().sum() == 0
)

malmem_source_index_pass = (
    (
        malmem_imbalance["source_index"] < 0
    ).all()
)

malmem_separation_pass = (
    malmem_imbalance_attack_types.isdisjoint(
        primary_attack_types
    )
)

malmem_validation = {
    "dataset": "CIC-MalMem-2022",
    "rows": len(malmem_imbalance),
    "expected_rows": 2342,
    "row_count_pass": malmem_rows_pass,
    "attack_type_pass": malmem_attack_only,
    "missing_value_pass": malmem_missing_pass,
    "source_index_pass": malmem_source_index_pass,
    "primary_attack_separation_pass": malmem_separation_pass,
    "exact_duplicates": int(
        malmem_imbalance.duplicated().sum()
    ),
    "feature_overlap_with_primary": malmem_feature_overlap
}

# ---------------- Android ----------------

android_attack_only = (
    android_imbalance["attack_type"].nunique() == 1
    and
    android_imbalance["attack_type"].iloc[0]
    == "class_imbalance_attack"
)

android_rows_pass = (
    len(android_imbalance) == 783
)

android_missing_pass = (
    android_imbalance.isna().sum().sum() == 0
)

android_source_index_pass = (
    (
        android_imbalance["source_index"] < 0
    ).all()
)

android_separation_pass = (
    android_imbalance_attack_types.isdisjoint(
        primary_attack_types_android
    )
)

android_validation = {
    "dataset": "Android",
    "rows": len(android_imbalance),
    "expected_rows": 783,
    "row_count_pass": android_rows_pass,
    "attack_type_pass": android_attack_only,
    "missing_value_pass": android_missing_pass,
    "source_index_pass": android_source_index_pass,
    "primary_attack_separation_pass": android_separation_pass,
    "exact_duplicates": int(
        android_imbalance.duplicated().sum()
    ),
    "feature_overlap_with_primary": android_feature_overlap
}

class_imbalance_validation = pd.DataFrame(
    [
        malmem_validation,
        android_validation
    ]
)

class_imbalance_validation["PHASE_2_10_PASS"] = (
    class_imbalance_validation[
        [
            "row_count_pass",
            "attack_type_pass",
            "missing_value_pass",
            "source_index_pass",
            "primary_attack_separation_pass"
        ]
    ].all(axis=1)
)

class_imbalance_validation.to_csv(
    results_dir / "table_class_imbalance_validation.csv",
    index=False
)

print("\n" + "=" * 70)
print("PHASE 2.10 — FINAL CLASS-IMBALANCE VALIDATION")
print("=" * 70)

print(
    class_imbalance_validation.to_string(
        index=False
    )
)

print("\nSaved:")
print(" - table_class_imbalance_validation.csv")


PHASE 2.10 — FINAL CLASS-IMBALANCE VALIDATION
        dataset  rows  expected_rows  row_count_pass  attack_type_pass  missing_value_pass  source_index_pass  primary_attack_separation_pass  exact_duplicates  feature_overlap_with_primary  PHASE_2_10_PASS
CIC-MalMem-2022  2342           2342            True              True                True               True                            True                 3                             0             True
        Android   783            783            True              True                True               True                            True                 0                             5             True

Saved:
 - table_class_imbalance_validation.csv


Dataset Identity & Provenance Documentation

Record dataset identities

In [110]:
# Phase 2.11.1 — Dataset identity summary

dataset_identity = pd.DataFrame([
    {
        "dataset_role": "Primary malware poisoning dataset",
        "local_file": "cic_malmem2022_poisoned.csv",
        "working_name": "CIC-MalMem-2022",
        "rows": len(malmem),
        "columns": len(malmem.columns),
        "substantive_numeric_features": len(malmem_numeric_features),
        "provisional_identity": "CIC-MalMem-2022"
    },
    {
        "dataset_role": "Primary Android poisoning dataset",
        "local_file": "cccs_andmal2020_poisoned.csv",
        "working_name": "Android",
        "rows": len(android),
        "columns": len(android.columns),
        "substantive_numeric_features": len(android_numeric_features),
        "provisional_identity": "CICMalDroid2020-structured feature file"
    },
    {
        "dataset_role": "Separate class-imbalance audit",
        "local_file": "cic_malmem2022_removed_audit.csv",
        "working_name": "CIC-MalMem-2022 class-imbalance audit",
        "rows": len(malmem_imbalance),
        "columns": len(malmem_imbalance.columns),
        "substantive_numeric_features": len(
            get_feature_columns_for_comparison(
                malmem_imbalance
            )
        ),
        "provisional_identity": "Separate class-imbalance artifact"
    },
    {
        "dataset_role": "Separate Android class-imbalance audit",
        "local_file": "cccs_andmal2020_removed_audit.csv",
        "working_name": "Android class-imbalance audit",
        "rows": len(android_imbalance),
        "columns": len(android_imbalance.columns),
        "substantive_numeric_features": len(
            get_feature_columns_for_comparison(
                android_imbalance
            )
        ),
        "provisional_identity": "Separate class-imbalance artifact"
    }
])

print(dataset_identity.to_string(index=False))

                          dataset_role                        local_file                          working_name  rows  columns  substantive_numeric_features                    provisional_identity
     Primary malware poisoning dataset       cic_malmem2022_poisoned.csv                       CIC-MalMem-2022 58596       60                            55                         CIC-MalMem-2022
     Primary Android poisoning dataset      cccs_andmal2020_poisoned.csv                               Android 11598      475                           470 CICMalDroid2020-structured feature file
        Separate class-imbalance audit  cic_malmem2022_removed_audit.csv CIC-MalMem-2022 class-imbalance audit  2342       59                            55       Separate class-imbalance artifact
Separate Android class-imbalance audit cccs_andmal2020_removed_audit.csv         Android class-imbalance audit   783      474                           470       Separate class-imbalance artifact


Compare Android file structure with CICMalDroid2020

In [111]:
# Phase 2.11.2 — Android identity comparison

android_identity_check = pd.DataFrame([
    {
        "property": "Rows",
        "local_value": len(android),
        "official_CICMalDroid2020_reference": 11598,
        "match": len(android) == 11598
    },
    {
        "property": "Substantive numeric features",
        "local_value": len(android_numeric_features),
        "official_CICMalDroid2020_reference": 470,
        "match": len(android_numeric_features) == 470
    },
    {
        "property": "Observed malware/benign categories",
        "local_value": android["Class"].nunique(),
        "official_CICMalDroid2020_reference": 5,
        "match": android["Class"].nunique() == 5
    }
])

print(
    android_identity_check.to_string(index=False)
)

                          property  local_value  official_CICMalDroid2020_reference  match
                              Rows        11598                               11598   True
      Substantive numeric features          470                                 470   True
Observed malware/benign categories            5                                   5   True


Compare Android file against official CCCS-CIC-AndMal scale

In [112]:
# Phase 2.11.3 — Android identity distinction

android_andmal_identity_check = pd.DataFrame([
    {
        "property": "Local Android rows",
        "local_value": len(android),
        "official_CCCS_CIC_AndMal_2020_reference": 400000,
        "matches_official_scale": len(android) == 400000
    },
    {
        "property": "Local Android class count",
        "local_value": android["Class"].nunique(),
        "official_CCCS_CIC_AndMal_2020_reference": 14,
        "matches_reference_category_count": (
            android["Class"].nunique() == 14
        )
    }
])

print(
    android_andmal_identity_check.to_string(index=False)
)

                 property  local_value  official_CCCS_CIC_AndMal_2020_reference matches_official_scale matches_reference_category_count
       Local Android rows        11598                                   400000                  False                              NaN
Local Android class count            5                                       14                    NaN                            False


Verify Android class structure

In [113]:
# Phase 2.11.4 — Android class identity audit

android_class_distribution = (
    android["Class"]
    .value_counts()
    .sort_index()
)

print("Android Class distribution:")
print(android_class_distribution)

print("\nNumber of classes:",
      android["Class"].nunique())

print("\nClass labels:")
print(
    android["Class"]
    .dropna()
    .unique()
)

Android Class distribution:
Class
Adware         1053
Banking        1764
Benign         3361
Riskware       2140
SMS malware    3280
Name: count, dtype: int64

Number of classes: 5

Class labels:
<StringArray>
['Benign', 'Adware', 'Banking', 'SMS malware', 'Riskware']
Length: 5, dtype: str


In [114]:
# Phase 2.11.5 — Feature-count provenance check

feature_identity = pd.DataFrame([
    {
        "dataset": "CIC-MalMem-2022",
        "local_substantive_features": len(
            malmem_numeric_features
        ),
        "expected_substantive_features": 55,
        "match": (
            len(malmem_numeric_features) == 55
        )
    },
    {
        "dataset": "Android / CICMalDroid2020 structure",
        "local_substantive_features": len(
            android_numeric_features
        ),
        "expected_substantive_features": 470,
        "match": (
            len(android_numeric_features) == 470
        )
    }
])

print(
    feature_identity.to_string(index=False)
)

                            dataset  local_substantive_features  expected_substantive_features  match
                    CIC-MalMem-2022                          55                             55   True
Android / CICMalDroid2020 structure                         470                            470   True


Create the formal provenance decision

In [115]:
# Phase 2.11.6 — Formal provenance decision

provenance_decision = pd.DataFrame([
    {
        "dataset": "CIC-MalMem-2022",
        "local_file": "cic_malmem2022_poisoned.csv",
        "identity_decision": "Retain CIC-MalMem-2022 working identity",
        "reason": (
            "Local structure matches the documented "
            "CIC-MalMem-2022 dataset configuration."
        )
    },
    {
        "dataset": "Android",
        "local_file": "cccs_andmal2020_poisoned.csv",
        "identity_decision": (
            "Document as CICMalDroid2020-structured "
            "Android feature dataset"
        ),
        "reason": (
            "Local file has 11,598 rows and 470 substantive "
            "features, matching the official CICMalDroid2020 "
            "CSV structure; it does not match the scale/category "
            "structure of the official CCCS-CIC-AndMal-2020 dataset."
        )
    },
    {
        "dataset": "Android class-imbalance artifact",
        "local_file": "cccs_andmal2020_removed_audit.csv",
        "identity_decision": (
            "Retain as separate class-imbalance audit artifact"
        ),
        "reason": (
            "Artifact is not part of the seven primary "
            "poisoning mechanisms."
        )
    }
])

print(
    provenance_decision.to_string(index=False)
)

                         dataset                        local_file                                              identity_decision                                                                                                                                                                                                     reason
                 CIC-MalMem-2022       cic_malmem2022_poisoned.csv                        Retain CIC-MalMem-2022 working identity                                                                                                                              Local structure matches the documented CIC-MalMem-2022 dataset configuration.
                         Android      cccs_andmal2020_poisoned.csv Document as CICMalDroid2020-structured Android feature dataset Local file has 11,598 rows and 470 substantive features, matching the official CICMalDroid2020 CSV structure; it does not match the scale/category structure of the official CCCS-CIC-AndMal-2020 dataset.
A

In [116]:
# Phase 2.11.7 — Save dataset identity/provenance artifacts

results_dir = Path("../results/tables")
docs_dir = Path("../docs")

results_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)

dataset_identity.to_csv(
    results_dir / "table_dataset_identity.csv",
    index=False
)

android_identity_check.to_csv(
    results_dir / "table_android_cicmaldroid_identity_check.csv",
    index=False
)

android_andmal_identity_check.to_csv(
    results_dir / "table_android_andmal_identity_check.csv",
    index=False
)

feature_identity.to_csv(
    results_dir / "table_dataset_feature_identity.csv",
    index=False
)

provenance_decision.to_csv(
    results_dir / "table_dataset_provenance_decision.csv",
    index=False
)


# Formal Markdown documentation

provenance_text = """# Dataset Identity and Provenance

## CIC-MalMem-2022

The primary memory-malware dataset is retained under the working
identity `CIC-MalMem-2022`.

Local audited structure:

- Rows: 58,596
- Columns: 59
- Substantive numeric features: 55
- Seven poisoning mechanisms plus clean observations are represented
  in the primary experimental file.

## Android dataset

The local Android poisoning dataset is stored as:

`cccs_andmal2020_poisoned.csv`

The audited local structure is:

- Rows: 11,598
- Columns: 474
- Substantive numeric features: 470
- Five observed malware/benign classes.

The local structure matches the documented CICMalDroid2020
configuration of 11,598 final samples and 470 extracted features.

Therefore, for thesis reporting, this file should NOT be described
as the official 400,000-sample CCCS-CIC-AndMal-2020 dataset.

The thesis should instead explicitly document the local Android
dataset as a CICMalDroid2020-structured 11,598-sample, 470-feature
Android dataset, while preserving the original local filename as
part of the provenance record.

## Separate class-imbalance artifacts

The two `removed_audit` files are retained separately and are not
merged into the primary seven poisoning mechanisms.

They are used only for the separate class-imbalance audit.

## Methodological decision

Raw dataset files are not renamed, modified, or overwritten during
this provenance audit. Dataset identity is documented separately
from the local filenames.
"""

provenance_file = docs_dir / "dataset_identity_and_provenance.md"

provenance_file.write_text(
    provenance_text,
    encoding="utf-8"
)

print("Saved Phase 2.11 provenance artifacts:")
print(" - results/tables/table_dataset_identity.csv")
print(" - results/tables/table_android_cicmaldroid_identity_check.csv")
print(" - results/tables/table_android_andmal_identity_check.csv")
print(" - results/tables/table_dataset_feature_identity.csv")
print(" - results/tables/table_dataset_provenance_decision.csv")
print(" - docs/dataset_identity_and_provenance.md")

Saved Phase 2.11 provenance artifacts:
 - results/tables/table_dataset_identity.csv
 - results/tables/table_android_cicmaldroid_identity_check.csv
 - results/tables/table_android_andmal_identity_check.csv
 - results/tables/table_dataset_feature_identity.csv
 - results/tables/table_dataset_provenance_decision.csv
 - docs/dataset_identity_and_provenance.md


In [117]:
# Phase 2.11.8 — Final dataset identity validation

validation_rows = [
    {
        "dataset": "CIC-MalMem-2022",
        "local_rows": len(malmem),
        "expected_rows": 58596,
        "row_count_pass": len(malmem) == 58596,

        "local_features": len(malmem_numeric_features),
        "expected_features": 55,
        "feature_count_pass": (
            len(malmem_numeric_features) == 55
        ),

        "identity_documented": True
    },

    {
        "dataset": "Android / CICMalDroid2020 structure",
        "local_rows": len(android),
        "expected_rows": 11598,
        "row_count_pass": len(android) == 11598,

        "local_features": len(android_numeric_features),
        "expected_features": 470,
        "feature_count_pass": (
            len(android_numeric_features) == 470
        ),

        "identity_documented": True
    }
]

dataset_identity_validation = pd.DataFrame(
    validation_rows
)

dataset_identity_validation["PHASE_2_11_PASS"] = (
    dataset_identity_validation[
        [
            "row_count_pass",
            "feature_count_pass",
            "identity_documented"
        ]
    ].all(axis=1)
)

dataset_identity_validation.to_csv(
    results_dir / "table_dataset_identity_validation.csv",
    index=False
)

print("\n" + "=" * 70)
print("PHASE 2.11 — FINAL DATASET IDENTITY VALIDATION")
print("=" * 70)

print(
    dataset_identity_validation.to_string(
        index=False
    )
)

print("\nSaved:")
print(" - table_dataset_identity_validation.csv")
print(" - docs/dataset_identity_and_provenance.md")


PHASE 2.11 — FINAL DATASET IDENTITY VALIDATION
                            dataset  local_rows  expected_rows  row_count_pass  local_features  expected_features  feature_count_pass  identity_documented  PHASE_2_11_PASS
                    CIC-MalMem-2022       58596          58596            True              55                 55                True                 True             True
Android / CICMalDroid2020 structure       11598          11598            True             470                470                True                 True             True

Saved:
 - table_dataset_identity_validation.csv
 - docs/dataset_identity_and_provenance.md


Data Dictionary & Variable Governance

Define variable roles

In [ ]:
print("=" * 70)
print("PHASE 2.12.1 — VARIABLE ROLE GOVERNANCE")
print("=" * 70)

metadata_columns = [
    "attack_type",
    "source_index",
    "poisoned",
    "original_label",
    "Class"
]

variable_governance = pd.DataFrame([
    {
        "variable": "attack_type",
        "role": "Attack mechanism metadata",
        "allowed_in_anomaly_model": False,
        "allowed_in_downstream_model": False,
        "allowed_for_evaluation": True,
        "reason": "Used only for attack-specific evaluation, stratification, and robustness analysis."
    },
    {
        "variable": "source_index",
        "role": "Source/provenance metadata",
        "allowed_in_anomaly_model": False,
        "allowed_in_downstream_model": False,
        "allowed_for_evaluation": True,
        "reason": "Used for provenance and duplication validation, not prediction."
    },
    {
        "variable": "poisoned",
        "role": "Evaluation target",
        "allowed_in_anomaly_model": False,
        "allowed_in_downstream_model": False,
        "allowed_for_evaluation": True,
        "reason": "Derived target: 1 when attack_type is not clean; never a model predictor."
    },
    {
        "variable": "original_label",
        "role": "Reference/original class label",
        "allowed_in_anomaly_model": False,
        "allowed_in_downstream_model": False,
        "allowed_for_evaluation": True,
        "reason": "Used for label-consistency analysis and reference information, not prediction."
    },
    {
        "variable": "Class",
        "role": "Observed class label",
        "allowed_in_anomaly_model": False,
        "allowed_in_downstream_model": True,
        "allowed_for_evaluation": True,
        "reason": "Used as the downstream malware-classification target, never as an anomaly predictor."
    }
])

display(variable_governance)